# 07a · Dynamics: does the EMG drive the hand over time, and what structure helps a decoder?

**Investigation ladder** (see [`EVALUATION.md`](../EVALUATION.md)):
- **Stage 5 · Dynamics: is movement input-driven, and what structure helps a decoder?**

**Read first:** Wu et al. 2006, *Bayesian population decoding of motor cortical activity using a Kalman filter* (Neural Computation); Almani, Lazzari, Walker & Saxena 2026, *Embodied sensorimotor control* (Annu Rev Biomed Eng), sections 4 (internal models, Kalman state estimation) and 5 (musculoskeletal models).

**We expect:** a model with **state** (the pose and its velocity) and the EMG as **input** beats a static EMG → pose map, and the EMG-driven dynamics beat the same dynamics with the input switched off. Structure that matches the body (a per-task resting posture, a few shared EMG–pose latents) should help most on stages the decoder never saw. **Wrong if:** 'no change' beats the EMG-driven rollouts, or no stateful decoder beats the median pose of the new task's first few seconds on unseen stages.

**This notebook is stage 5, part A** (part B, [`07b`](07b_compositional_dynamics.ipynb), adds compositional models
in the sense of Lazzari & Saxena 2025: switching dynamical motifs and a multitask RNN). Five parts, each building on
the last:

| Part | Model | Question |
|---|---|---|
| A | linear dynamics `z(t+1) = A z(t) + B u(t)` | Does adding EMG help predict the next pose? |
| B | Kalman filter (pose = hidden state, EMG = observation) | Can a model with state decode the pose? |
| C | **physical (grey-box) model**: activation dynamics + a spring-damper hand | Does building in the body's physics help? |
| D | shared dynamics + a **per-task resting posture** | Does a task-specific equilibrium fix unseen stages? |
| E | dynamics in the **shared EMG–pose subspace** (notebook 06) | Are 2–3 shared latents enough to carry the dynamics? |

**The data is version 2** (notebook 01, Task 5b): the pose is low-passed at 6 Hz and tracking dropouts are padded.
Some numbers in earlier write-ups were measured on the unsmoothed pose and will differ here.

**How this notebook works** (the same in every notebook)
- **`# TODO`** cells are yours. Each has a docstring with the input and output shapes. You write the core
  algorithms yourself in NumPy.
- Under every TODO there are three **💡 hints** that open one at a time: the plan in words, the tools with a tiny
  example, then a skeleton with blanks (`____`).
- After each TODO a **✅ check cell** compares your answer with the reference in [`ref07a.py`](ref07a.py), then a **📊 plot**
  shows what your code did. Other notebooks use the reference, so your blanks never break theirs.
- **✍️ questions** are for your own words, in the markdown cell under them.
- The **Result** cell at the end is the **gate**: it decides whether the next stage of the investigation is worth running.

## 🎯 Goal and milestones
*Draft milestones: agree them as a team on Friday, then edit here.*

**Proposal:** ➕ beyond the proposal, but it decides stage 6: the bottleneck comparison is only meaningful with a decoder that works.

**Goal:** find out whether the hand's movement is input-driven, and which structure (state, physics, a per-task resting posture, shared latents) gives a decoder that beats the honest baselines, within recordings and on unseen stages.

**Why:** notebook 05 showed that an instantaneous EMG → pose map barely beats each recording's own typical posture; position is the *result* of force over time, so a decoder probably needs state.

**Milestones**
1. Rollouts: driven vs input-off vs 'no change' (Parts A and C)
2. Within recordings: every decoder against the per-recording median pose, with block CIs
3. Unseen stages with 5 s of calibration: every decoder against the calibration median, with CIs over folds
4. The ✍️ questions answered, and the gate filled in

**Gate (decides the next stage):** the EMG-driven model beats the input-off model in free rollouts, **and** a stateful decoder beats the median pose of the new task's first 5 s on unseen stages, with a 95% CI that excludes 0.

**Done when:** the gate cell is filled in with numbers and CIs, and you can say which structure helped and which didn't.

In [ ]:
# ── Setup: works on Colab and locally ─────────────────────────────────────────
# After this cell the working directory is the repo root, wherever you run from.
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ or Path("/content").exists()
if IN_COLAB:
    ROOT = Path("/content/emg2pose")
    if not ROOT.exists():
        !git clone https://github.com/Wilbertbh-Tan/emg2pose.git {ROOT}
    else:
        !git -C {ROOT} pull -q                 # pick up files pushed since the first clone
    !pip install -q h5py
else:
    ROOT = Path.cwd().resolve()
    while not (ROOT / "setup.py").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    assert (ROOT / "setup.py").exists(), "Run this notebook from inside the emg2pose repo"
os.chdir(ROOT)

_d = ROOT / "notebooks" / "experiments" / "07_dynamics"
assert (_d / "ref07a.py").exists(), f"{_d}/ref07a.py is missing: pull the latest version of the repo"
sys.path.insert(0, str(_d))
sys.modules.pop("ref07a", None)
_d = ROOT / "notebooks" / "experiments" / "06_shared_subspace"
assert (_d / "ref06.py").exists(), f"{_d}/ref06.py is missing: pull the latest version of the repo"
sys.path.insert(0, str(_d))
sys.modules.pop("ref06", None)
_d = ROOT / "notebooks" / "experiments" / "02_how_many_synergies"
assert (_d / "ref02.py").exists(), f"{_d}/ref02.py is missing: pull the latest version of the repo"
sys.path.insert(0, str(_d))
sys.modules.pop("ref02", None)
_d = ROOT / "notebooks" / "experiments" / "common"
sys.path.insert(0, str(_d)); sys.modules.pop("protocol", None)
import protocol                 # shared evaluation protocol: sessions, folds, baselines, nulls, CIs
import ref07a as ref          # the reference answers, used only by the ✅ check cells

PROCESSED = ROOT / "data" / "processed_mini.npz"   # made by notebook 01's prep.py (data/ is never committed)

In [ ]:
# ── Load the shared data (rebuilds it with prep.py if missing or built by an older version) ──
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, str(ROOT / "notebooks" / "experiments" / "01_data_prep")); sys.modules.pop("prep", None)
import prep
if not prep.is_current(PROCESSED):
    subprocess.run([sys.executable, "notebooks/experiments/01_data_prep/prep.py"], check=True)
d = np.load(PROCESSED)
import ref02, ref06              # notebooks 02 (NMF/PCA) and 06 (shared EMG–pose subspace): or paste your own

emg        = d["emg"].astype(np.float64)   # (N, 16)  EMG envelope, 50 samples/s, non-negative
pose       = d["pose"].astype(np.float64)  # (N, 20)  joint angles in degrees (low-passed at 6 Hz)
pose_valid = d["pose_valid"]               # (N,)     False where pose tracking failed (dropouts padded)
rec        = d["rec"]                      # (N,)     which recording (0–29) each row comes from
split      = d["split"]                    # (N,)     0 = train, 1 = test, -1 = gap (use neither)
stage_of_rec, side_of_rec = d["stage"], d["side"]   # (30,) task name and hand per recording
filename   = d["filename"]                # (30,) file name per recording (its number pairs the two hands)
JOINT_NAMES = [
    "THUMB_CMC_FE", "THUMB_CMC_AA", "THUMB_MCP_FE", "THUMB_IP_FE",
    "INDEX_MCP_AA", "INDEX_MCP_FE", "INDEX_PIP_FE", "INDEX_DIP_FE",
    "MIDDLE_MCP_AA", "MIDDLE_MCP_FE", "MIDDLE_PIP_FE", "MIDDLE_DIP_FE",
    "RING_MCP_AA", "RING_MCP_FE", "RING_PIP_FE", "RING_DIP_FE",
    "PINKY_MCP_AA", "PINKY_MCP_FE", "PINKY_PIP_FE", "PINKY_DIP_FE",
]
train, test = split == 0, split == 1
SEED, K_SYN, K_POSE = 0, 4, 10

# Ready-made inputs for Part C: synergies and eigenpostures (notebook 02), fitted on training frames only
_, W, _ = ref02.nmf_fit(emg[train], K_SYN, seed=SEED)
C = ref02.nmf_transform(emg, W, seed=SEED)            # (N, 4)  synergy activations
U_syn = np.log(C + 1e-3)                              # (N, 4)  log activations: the physical model's input
LE = np.log(emg + 1e-3)                               # (N, 16) log EMG: what the Kalman filters observe
pose_mean, pose_comps = ref02.pca_fit(pose[train & pose_valid], K_POSE)
Qs = ref02.pca_transform(pose, pose_mean, pose_comps)      # (N, 10) eigenposture scores
to_angles = lambda q: ref02.pca_inverse(q, pose_mean, pose_comps)   # (T, 10) scores → (T, 20) degrees

# Ready-made input for Part E: log EMG from 100 and 200 ms earlier (notebook 06's lag features)
X_lag = ref06.lag_features(LE, rec, [5, 10])          # (N, 32)
print(f"{len(emg)} frames | {train.sum()} train | {test.sum()} test | {len(stage_of_rec)} recordings")

### The variables you start with
`N` = 105,484 frames (about 35 minutes at 50 frames/s, one person). Rows are frames, 20 ms apart. Boolean arrays are
**masks**: `emg[train]` keeps only the train rows.

| Name | Shape | What it is |
|---|---|---|
| `emg`, `pose`, `pose_valid`, `rec`, `split`, `train`, `test` | | as in notebooks 02–06 |
| `stage_of_rec`, `side_of_rec`, `filename` | (30,) | task, hand and file name per recording. The **recording number in the file name** pairs the two recordings made at the same time (one block); pass `filename=filename` to every `protocol` function |
| `LE` | (N, 16) | log EMG envelope: what the Kalman filters **observe** |
| `U_syn` | (N, 4) | log synergy activations: the physical model's **input** (Part C) |
| `Qs`, `to_angles` | (N, 10) | eigenposture scores and the map back to 20 angles (Part C) |
| `X_lag` | (N, 32) | log EMG from 100 and 200 ms earlier (Part E) |

**Why logs?** EMG noise grows with activity (*signal-dependent noise*). A log makes the noise roughly the same size at
low and high activity, which suits least squares. Notebook 05 found log-EMG with history was its best decoder.

**Rows, not columns.** In the maths a state is a column vector and the model reads `z(t+1) = A z(t)`. In code each
frame is a **row**, so the same model for many frames at once reads `Z_next = Z @ A.T`. Keep this in mind everywhere.

Letters below: **M** = usable steps t → t+1, **T** = frames in the array you pass in, **H** = steps ahead,
**n** = pose dimensions of a model's state, **k** = synergies.

## Before you run anything: your prediction

Write what you expect and why (2–3 sentences). Compare at the end. Surprises are the interesting part.

> *Prediction:*

---
# Part A · Linear dynamics: does adding EMG help predict the next pose?

## Task 1 · Collect the steps t → t+1
To learn how the pose changes from one frame to the next, we need many examples of
`(pose now, EMG now, pose next)`. A step t → t+1 is only usable if:
- **both** frames are allowed by the mask (e.g. training frames with valid pose), and
- both come from the **same recording**: the last frame of recording 3 followed by the first frame of recording 4
  is not a real movement.

A tracking dropout or the train/test gap therefore breaks the chain: the step *into* it and the step *out of* it are
both dropped, so the model never learns a fake jump.

In [ ]:
def make_triples(pose: np.ndarray, emg: np.ndarray, rec: np.ndarray,
                 mask: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Collect every step t → t+1 where both frames are usable and in the same recording.

    Parameters
    ----------
    pose : (N, 20) float, joint angles (degrees): the state z
    emg  : (N, 16) float, EMG envelope: the input u
    rec  : (N,) int, recording of each frame
    mask : (N,) bool, frames allowed (e.g. train & pose_valid)

    Returns
    -------
    Z      : (M, 20) float, z(t)
    U      : (M, 16) float, u(t)
    Z_next : (M, 20) float, z(t+1)
    M = number of t where mask[t] and mask[t+1] are True and rec[t] == rec[t+1].
    """
    # TODO
    raise NotImplementedError


Z, U, Z_next = None, None, None   # TODO: make_triples(pose, emg, rec, train & pose_valid)

#### 💡 Stuck on Task 1? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. Line every frame up with the frame after it: compare `mask[:-1]` (frames 0 … N−2) with `mask[1:]` (frames 1 … N−1).
2. A step t is usable if `mask[t]`, `mask[t+1]` and "same recording" are all True: combine three (N−1,) masks with `&`.
3. Find the positions `t` where that is True, then pick rows `t` (now) and `t + 1` (next).

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `a[:-1]` = all but the last entry, `a[1:]` = all but the first. Side by side they pair entry t with entry t+1.
- `np.flatnonzero(m)` → the positions where a boolean array is True.

```python
mask = np.array([True, True, False, True, True]); rec = np.array([0, 0, 0, 0, 1])
ok = mask[:-1] & mask[1:] & (rec[:-1] == rec[1:])   # [ True, False, False, False]
np.flatnonzero(ok)                                  # array([0]): only step 0 → 1 is usable
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def make_triples(pose, emg, rec, mask):
    ok = mask[:-1] & mask[____] & (rec[:-1] == rec[____])
    t = np.flatnonzero(____)
    return pose[t], emg[____], pose[t + ____]

Z, U, Z_next = make_triples(pose, emg, rec, train & pose_valid)
```

</details>

In [ ]:
# ✅ Check Task 1
mask_toy = np.array([True, True, False, True, True, True]); rec_toy = np.array([0, 0, 0, 0, 0, 1])
p_toy = np.arange(6.0)[:, None] * np.ones((1, 20)); e_toy = np.arange(6.0)[:, None] * np.ones((1, 16))
Zt, Ut, Znt = make_triples(p_toy, e_toy, rec_toy, mask_toy)
assert Zt[:, 0].tolist() == [0.0, 3.0] and Znt[:, 0].tolist() == [1.0, 4.0], \
    f"toy: steps from frames {Zt[:, 0].tolist()}, expected [0, 3] (not across the masked frame or the recording change)"
r_Z, r_U, r_Zn = ref.make_triples(pose, emg, rec, train & pose_valid)
assert Z.shape == r_Z.shape, f"Z shape {Z.shape}, expected {r_Z.shape}"
assert np.array_equal(Z, r_Z) and np.array_equal(U, r_U) and np.array_equal(Z_next, r_Zn)
print(f"✅ {len(Z):,} usable training steps")

✍️ **Q1.** Why must a step t → t+1 never cross a tracking dropout, the gap between train and test, or the boundary between two recordings? What would one such fake step look like to the least-squares fit?

> *Your answer:*

## Task 2 · Least squares from scratch
Almost everything in this notebook is one problem: find the matrix `Θ` that makes `X Θ` as close as possible to `Y`.
`X` (M, p) holds the inputs, one row per example; `Y` (M, q) the outputs we want to predict.

**The idea.** "Close" means the smallest total squared error `‖XΘ − Y‖²`. At the best `Θ`, the leftover error
`XΘ − Y` has nothing in common with any input column: if it did, you could shift `Θ` a little in that direction and
do better. "Nothing in common" means `Xᵀ(XΘ − Y) = 0`, which rearranges into the **normal equations**:

$$(X^\top X)\,\Theta = X^\top Y \qquad\Rightarrow\qquad \Theta = (X^\top X)^{-1} X^\top Y$$

`XᵀX` is only (p, p) and `XᵀY` is (p, q): small, whatever M is. In NumPy, **solve** the system rather than inverting:
`np.linalg.solve(X.T @ X, X.T @ Y)` is faster and more accurate than `np.linalg.inv(...)`.

**The bias column.** Joint angles aren't centred on 0 (a relaxed hand sits around 20–40° flexed), so the model
needs a constant offset `b`: `z(t+1) = A z(t) + b`. The trick: add a column of **ones** to `X`. Its coefficient
is multiplied by 1 in every row, so least squares learns it as the offset, with no extra code.

In [ ]:
def add_bias(X: np.ndarray) -> np.ndarray:
    """(M, p) → (M, p + 1): X with a column of ones added on the RIGHT."""
    # TODO
    raise NotImplementedError


def least_squares(X: np.ndarray, Y: np.ndarray) -> np.ndarray:
    """Θ (p, q) minimising the squared error ||X @ Θ − Y||², via the normal equations.

    Parameters
    ----------
    X : (M, p) float, inputs, one row per example
    Y : (M, q) float, outputs, one row per example

    Returns
    -------
    Theta : (p, q) float. Solve (Xᵀ X) Θ = Xᵀ Y with np.linalg.solve (not np.linalg.lstsq, not inv).
    """
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 2? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. `add_bias`: make an (M, 1) column of ones and stick it to the right of `X`.
2. `least_squares`: build the two small matrices `X.T @ X` and `X.T @ Y`, then hand both to `np.linalg.solve`.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.ones((M, 1))` → a column of ones. `np.hstack([A, B])` glues arrays side by side (same number of rows).
- `np.linalg.solve(K, R)` finds `Θ` with `K @ Θ = R`.

```python
X = np.array([[1.0], [2.0], [3.0]]); Y = 2 * X + 1          # y = 2x + 1
Xb = np.hstack([X, np.ones((3, 1))])
np.linalg.solve(Xb.T @ Xb, Xb.T @ Y)                         # [[2.], [1.]]: slope 2, offset 1
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def add_bias(X):
    return np.hstack([X, np.ones((len(____), 1))])

def least_squares(X, Y):
    return np.linalg.solve(____ @ ____, ____ @ ____)
```

</details>

In [ ]:
# ✅ Check Task 2
assert add_bias(np.zeros((3, 2))).tolist() == [[0, 0, 1]] * 3, "add_bias: ones go in a new LAST column"
rng = np.random.default_rng(0)
X_toy = rng.normal(size=(200, 5)); Theta_true = rng.normal(size=(5, 3))
Y_toy = X_toy @ Theta_true + 0.1 * rng.normal(size=(200, 3))
Theta = least_squares(X_toy, Y_toy)
assert Theta.shape == (5, 3), f"shape {Theta.shape}, expected (5, 3)"
assert np.allclose(Theta, np.linalg.lstsq(X_toy, Y_toy, rcond=None)[0]), "differs from NumPy's own least squares"
print(f"✅ Least squares works: largest difference from the true Θ = {np.abs(Theta - Theta_true).max():.3f} (just noise)")

## Task 3 · Fit the dynamics with and without EMG
**With EMG:** `z(t+1) = A z(t) + B u(t) + b`. In rows: `Z_next ≈ Z @ A.T + U @ B.T + b`.
Put the inputs side by side, `[Z, U, 1]` (M, 20 + 16 + 1), and solve one least-squares problem. The answer `Θ`
is (37, 20): its first 20 rows are `A.T`, the next 16 are `B.T`, and the last row is `b`.

**Without EMG:** `z(t+1) = A z(t) + b`, from `[Z, 1]` (M, 21) alone. This is the control: if B adds nothing,
the EMG doesn't help predict where the hand goes next.

One thing to look at afterwards: `A` is close to the identity matrix. At 50 frames per second the hand barely moves
between frames, so the best one-step guess is "almost where it is now".

In [ ]:
def fit_AB(Z: np.ndarray, U: np.ndarray, Z_next: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Least-squares fit of z(t+1) = A z(t) + B u(t) + b.

    Parameters
    ----------
    Z : (M, 20), U : (M, 16), Z_next : (M, 20), from make_triples

    Returns
    -------
    A : (20, 20), B : (20, 16), b : (20,)
    """
    # TODO
    raise NotImplementedError


def fit_A(Z: np.ndarray, Z_next: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Least-squares fit of z(t+1) = A z(t) + b (no EMG). Returns A (20, 20) and b (20,)."""
    # TODO
    raise NotImplementedError


A_full, B_full, b_full = None, None, None   # TODO: fit_AB(Z, U, Z_next)
A_only, b_only = None, None                 # TODO: fit_A(Z, Z_next)

#### 💡 Stuck on Task 3? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. `fit_AB`: stack `[Z, U]` side by side, add the bias column, call your `least_squares` with `Z_next` as `Y`.
2. Cut `Θ` (37, 20) into its three blocks of rows: rows 0–19, rows 20–35, row 36. Transpose the first two
   (rows-vs-columns: the block is `A.T`, so `A` is its transpose).
3. `fit_A`: the same with `Z` alone; `Θ` is (21, 20).

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- Row slices: `theta[:20]`, `theta[20:-1]`, `theta[-1]` (the last row, as a (20,) vector). `.T` transposes.

```python
theta = np.arange(12.0).reshape(4, 3)    # pretend: 2 rows of A.T, 1 row of B.T, 1 row of b
theta[:2].T.shape, theta[2:-1].T.shape, theta[-1].shape   # ((3, 2), (3, 1), (3,))
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def fit_AB(Z, U, Z_next):
    theta = least_squares(add_bias(np.hstack([____, ____])), ____)
    nz = Z.shape[1]
    return theta[:nz].T, theta[nz:____].T, theta[____]

def fit_A(Z, Z_next):
    theta = least_squares(add_bias(____), Z_next)
    return theta[:-1].____, theta[-1]

A_full, B_full, b_full = fit_AB(Z, U, Z_next)
A_only, b_only = fit_A(Z, Z_next)
```

</details>

In [ ]:
# ✅ Check Task 3
r_A, r_B, r_b = ref.fit_AB(Z, U, Z_next)
assert A_full.shape == (20, 20) and B_full.shape == (20, 16) and b_full.shape == (20,), \
    f"shapes {A_full.shape}, {B_full.shape}, {b_full.shape}; expected (20, 20), (20, 16), (20,)"
assert np.allclose(A_full, r_A) and np.allclose(B_full, r_B) and np.allclose(b_full, r_b), \
    "values differ: check the order [Z, U, 1] and the transposes"
r_A0, r_b0 = ref.fit_A(Z, Z_next)
assert np.allclose(A_only, r_A0) and np.allclose(b_only, r_b0), "fit_A differs from the reference"
print(f"✅ Fitted. Diagonal of A (how much each joint keeps of itself): mean {np.diag(A_full).mean():.3f}")

In [ ]:
# 📊 What A and B look like
fig, ax = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={"width_ratios": [20, 16]})
im = ax[0].imshow(A_full, cmap="RdBu_r", vmin=-1, vmax=1)
ax[0].set_title("A: pose(t) → pose(t+1)"); ax[0].set_yticks(range(20), JOINT_NAMES, fontsize=6)
plt.colorbar(im, ax=ax[0], fraction=0.046)
lim = np.abs(B_full).max()
im = ax[1].imshow(B_full, cmap="RdBu_r", vmin=-lim, vmax=lim)
ax[1].set_title("B: EMG(t) → pose(t+1), degrees per unit envelope"); ax[1].set_xlabel("electrode")
plt.colorbar(im, ax=ax[1], fraction=0.046)
plt.tight_layout(); plt.show()

✍️ **Q2.** The normal equations solve `XᵀX θ = XᵀY`. When would `XᵀX` be nearly impossible to invert, and what does `np.linalg.lstsq` do differently? (Think of two EMG channels that always move together.)

> *Your answer:*

## Task 4 · Free rollout: run each model forward on its own predictions
One step ahead is easy (the hand barely moves in 20 ms). The real test is **running forward**: start from a real
test pose, predict the next one, then feed that *prediction* back in, again and again, for up to 25 steps (0.5 s).
Errors build up, and a model that understands how the hand moves should drift away from the truth more slowly.

Three models:
- **No change:** the pose stays where it started. The baseline to beat.
- **A only:** `z ← A z + b`, using nothing but the pose.
- **A and B:** `z ← A z + B u(t) + b`, also fed the real EMG at each step.

**Starting points** must have 25 usable steps ahead: frames t, t+1, …, t+25 all test frames with valid pose, all in
the same recording. To keep it fast we use every 10th such starting point.

In [ ]:
def rollout(z0: np.ndarray, U_future: np.ndarray, A: np.ndarray, b: np.ndarray,
            B: np.ndarray | None = None) -> np.ndarray:
    """Run the model forward, feeding it its own predictions.

    Parameters
    ----------
    z0       : (20,) float, the real starting pose
    U_future : (H, 16) float, the real EMG at the start frame and the H − 1 frames after it
    A        : (20, 20); b : (20,)
    B        : (20, 16), or None for the model without EMG

    Returns
    -------
    Z_pred : (H, 20) float, predicted pose after 1, 2, …, H steps.
    Step h uses U_future[h]: z(t+h+1) = A z(t+h) + B u(t+h) + b.
    """
    # TODO
    raise NotImplementedError


def rollout_starts(mask: np.ndarray, rec: np.ndarray, horizon: int) -> np.ndarray:
    """Frames t where frames t, t+1, …, t+horizon are all allowed by mask and in the same recording.

    Returns (S,) int, sorted. Hint: a step t → t+1 is usable exactly as in make_triples.
    """
    # TODO
    raise NotImplementedError


H_MAX = 25
starts = None   # TODO: rollout_starts(test & pose_valid, rec, H_MAX)[::10]

errors: dict[str, np.ndarray] = {}   # model → (H_MAX,) mean angular error (°) after 1, 2, …, H_MAX steps
# TODO: for each start t, the true future is pose[t + 1 : t + 1 + H_MAX] (H_MAX, 20); the EMG is emg[t : t + H_MAX]
# TODO: errors["no change"]: the prediction is pose[t] at every step
# TODO: errors["A only"] = rollout(..., A_only, b_only);  errors["A and B"] = rollout(..., A_full, b_full, B_full)
# For each model: the mean over starts and joints of |prediction − truth|, one number per step → shape (H_MAX,)

#### 💡 Stuck on Task 4? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. `rollout`: copy `z0`, then loop `h = 0 … H−1`: apply the model once, store the result in row `h`, carry on
   from it. Skip the `B` term when `B is None`.
2. `rollout_starts`: build the "step t → t+1 is usable" mask from Task 1. A start needs `horizon` usable steps in a
   row. The simplest version: a loop that checks `ok[t : t + horizon].all()`. The fast version: a running sum
   with `np.convolve`.
3. Errors: for each model, make an array (S, H_MAX) of per-step errors (the mean over the 20 joints), then average
   over starts with `.mean(axis=0)`.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.empty((H, 20))` makes an array to fill row by row. `z = A @ z + b` applies the model to one (20,) pose.
- `np.tile(p, (H, 1))` repeats a (20,) pose H times → (H, 20), the "no change" prediction.
- Running sum: `np.convolve(ok, np.ones(n), mode="valid")[t]` = `ok[t : t + n].sum()`.

```python
ok = np.array([1, 1, 1, 0, 1, 1])
np.convolve(ok, np.ones(2), mode="valid")   # [2, 2, 1, 1, 2]: entry t = ok[t] + ok[t+1]
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def rollout(z0, U_future, A, b, B=None):
    z = z0.copy()
    out = np.empty((len(U_future), len(z0)))
    for h in range(len(U_future)):
        z = A @ ____ + b + (B @ U_future[____] if B is not None else 0.0)
        out[h] = ____
    return out

def rollout_starts(mask, rec, horizon):
    ok = mask[:-1] & mask[1:] & (rec[:-1] == rec[1:])
    good = [t for t in range(len(ok) - horizon + 1) if ok[t : t + ____].all()]
    return np.array(good, dtype=int)

starts = rollout_starts(test & pose_valid, rec, H_MAX)[::10]
predict = {
    "no change": lambda t: np.tile(pose[t], (H_MAX, 1)),
    "A only":    lambda t: rollout(pose[t], emg[t : t + H_MAX], A_only, b_only),
    "A and B":   lambda t: rollout(pose[t], emg[t : t + H_MAX], ____, ____, ____),
}
for name, f in predict.items():
    per_start = np.array([np.abs(f(t) - pose[t + 1 : t + 1 + H_MAX]).mean(axis=1) for t in starts])  # (S, H_MAX)
    errors[name] = per_start.mean(axis=____)
```

</details>

In [ ]:
# ✅ Check Task 4
A_toy = 0.5 * np.eye(20); b_toy = np.ones(20)
out = rollout(np.zeros(20), np.zeros((3, 16)), A_toy, b_toy)
assert out.shape == (3, 20) and np.allclose(out[:, 0], [1.0, 1.5, 1.75]), \
    f"toy rollout gives {out[:, 0]}, expected [1, 1.5, 1.75] (each step feeds in the previous prediction)"
r_starts = ref.rollout_starts(test & pose_valid, rec, H_MAX)[::10]
assert np.array_equal(starts, r_starts), f"{len(starts)} starts, expected {len(r_starts)}"
t0 = r_starts[0]
r_roll = ref.rollout(pose[t0], emg[t0:t0 + H_MAX], A_full, b_full, B_full)
assert np.allclose(rollout(pose[t0], emg[t0:t0 + H_MAX], A_full, b_full, B_full), r_roll), "rollout with B differs"
assert set(errors) == {"no change", "A only", "A and B"} and all(e.shape == (H_MAX,) for e in errors.values())
r_models = {"no change": lambda t: np.tile(pose[t], (H_MAX, 1)),
            "A only": lambda t: ref.rollout(pose[t], emg[t:t + H_MAX], A_only, b_only),
            "A and B": lambda t: ref.rollout(pose[t], emg[t:t + H_MAX], A_full, b_full, B_full)}
for name, f in r_models.items():
    r_e = np.mean([np.abs(f(t) - pose[t + 1:t + 1 + H_MAX]).mean(axis=1) for t in r_starts], axis=0)
    assert np.allclose(errors[name], r_e), f"errors[{name!r}] differ from the reference"
print("✅ Rollout works. Mean error (°) after 1 / 5 / 25 steps:")
for name, e in errors.items():
    print(f"   {name:10s} {e[0]:5.2f}  {e[4]:5.2f}  {e[24]:5.2f}")

### 📊 How fast does each model drift?
Error against the number of steps ahead. A model that has learned something should stay **below** "no change".
Look carefully before you trust the expectation in the title cell, and write in the Result cell what you see.

In [ ]:
steps = np.arange(1, H_MAX + 1)
fig, ax = plt.subplots(figsize=(7, 4))
for name, e in errors.items():
    ax.plot(steps / 50 * 1000, e, marker="o", ms=3, label=name)
ax.set_xlabel("time ahead (ms)"); ax.set_ylabel("mean angular error (°)"); ax.legend()
ax.set_title("Free rollout on test data")
plt.tight_layout(); plt.show()

✍️ **Q4.** Does the EMG model beat "no change"? Does A alone? If not, why might a linear model that fits one step
well still drift *faster* than doing nothing? (Hint: look at what `A` does to a pose over 25 steps: compute the
largest `abs(np.linalg.eigvals(A_full))`, and ask where `A^25 z + …` ends up.)

> *Your answer:*

✍️ **Q3.** Compare the one-step error of A alone with A and B (Task 3 printout). How much does the EMG add for predicting just one frame (20 ms) ahead? Why might one step be an unfair test of whether EMG *drives* the hand?

> *Your answer:*

---
# Part B · A Kalman filter decoder
Part A asked "does EMG help *predict the future* pose?". A decoder asks something different: **given the EMG so far,
where is the hand *now*?** The Kalman filter answers it by combining two imperfect sources of information:

| Source | Model | What it knows |
|---|---|---|
| **Dynamics** (prior) | `z(t+1) = A z(t) + w`, noise `w ~ N(0, Q)` | the hand moves smoothly: the next pose is near the last one |
| **Observation** (EMG) | `y(t) = H z(t) + v`, noise `v ~ N(0, R)` | each pose comes with a typical EMG pattern |

Note the direction of the second line: it describes how the **pose produces the EMG** (a *generative* model,
`H` is 16 × 20), not how to read the pose off the EMG. The filter inverts it for you, in a principled way.

**At every frame it does two steps:**
1. **Predict.** Move the last estimate forward with the dynamics: `z⁻ = A z`. The uncertainty grows:
   `P⁻ = A P Aᵀ + Q`. (`P` is a 20 × 20 covariance: how unsure we are about each joint and pairs of joints.)
2. **Update.** Compare the EMG we *see*, `y(t)`, with the EMG the prediction *expects*, `H z⁻`. The difference
   (the *innovation*) nudges the estimate: `z = z⁻ + K (y − H z⁻)`, and the uncertainty shrinks: `P = (I − K H) P⁻`.

**The Kalman gain `K = P⁻ Hᵀ (H P⁻ Hᵀ + R)⁻¹`** decides how big the nudge is. Read it as a ratio:
*uncertainty about the pose* versus *noise in the EMG*. If the prediction is very uncertain (`P⁻` large) or the EMG is
clean (`R` small), `K` is large and the filter trusts the EMG. If the EMG is noisy, `K` is small and it trusts the
dynamics. Under the model's assumptions (linear, Gaussian noise) this is the best possible estimate.

**Two connections to what we discussed before**
- The filter is **causal**: frame t only uses EMG up to frame t, so it could run live. But our envelope was made with
  **zero-phase** filtering, which already mixes in about 100 ms of *future* EMG. So these results are slightly
  optimistic for a real live decoder; to be strict, rebuild the envelope with a causal filter.
- **One decoder per recording.** One linear map for all 30 recordings (both hands, 15 tasks) decodes barely better
  than always guessing the average pose. Brain–computer interfaces are calibrated per session for the same reason,
  so here each recording gets its own decoder: fit on its first 80%, decode its last 20%.

## Task 5 · Fit the Kalman model with least squares
Work with **centred** data: subtract the training mean of the pose and of the EMG first. Then no offsets are needed
and the plain equations above hold. Each part is a least-squares fit you have already written, plus the
**covariance of what it gets wrong**:

- **A, Q** from consecutive pairs `(Z, Z_next)`: `A = least_squares(Z, Z_next).T`; residuals `E = Z_next − Z @ A.T`;
  `Q = Eᵀ E / M`. Q says how much the hand moves in ways the dynamics can't predict.
- **H, R** from same-frame pairs `(Z_obs, Y_obs)`: `H = least_squares(Z_obs, Y_obs).T`; residuals
  `V = Y_obs − Z_obs @ H.T`; `R = Vᵀ V / T`. R says how much of the EMG the pose can't explain.

`Eᵀ E / M` is the covariance of the residuals (they average 0 because the data are centred).

#### Before Task 5: keep the dynamics stable
The pose is smooth (it was low-passed at 6 Hz in notebook 01), so the next frame is *very* predictable from the
current one. Least squares then sometimes returns an `A` with an eigenvalue slightly **above 1**, an unstable
model: any error grows a little every frame. Over a test block of a thousand frames, the filter's state and its
uncertainty `P` then blow up (you'd see a "singular matrix" error).

The fix is a standard one: **shrink every eigenvalue with |λ| > 0.995 back to 0.995**, keeping its direction. The
ready-made `ref.stabilise(A)` does exactly that (eigendecomposition, shrink, rebuild; conjugate pairs stay together
so `A` stays real). **Return `ref.stabilise(A)` instead of `A` at the end of `fit_kalman`** (compute `Q` from the
raw fit's residuals first). Every Kalman model in this notebook goes through `fit_kalman`, so they all inherit it.

In [ ]:
def fit_kalman(Z: np.ndarray, Z_next: np.ndarray, Z_obs: np.ndarray, Y_obs: np.ndarray
               ) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Fit z(t+1) = A z(t) + w (cov Q) and y(t) = H z(t) + v (cov R) on CENTRED data.

    Parameters
    ----------
    Z, Z_next : (M, 20) float, centred pose at t and t+1 (consecutive usable frames, e.g. from make_triples)
    Z_obs     : (T, 20) float, centred pose at usable frames
    Y_obs     : (T, 16) float, centred EMG at the same frames

    Returns
    -------
    A : (20, 20), Q : (20, 20), H : (16, 20), R : (16, 16)
    Return ref.stabilise(A) (see the note above); compute Q from the raw A's residuals.
    """
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 5? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. `A`: least squares from `Z` to `Z_next` (no bias column: the data are centred), transposed.
2. `Q`: what `A` gets wrong, `Z_next − Z @ A.T`, then its covariance: residualsᵀ @ residuals / number of rows.
3. `H` and `R`: the same two steps from `Z_obs` to `Y_obs`.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- Covariance of rows that average 0: `E.T @ E / len(E)` → (columns, columns).

```python
E = np.array([[1.0, -1.0], [-1.0, 1.0]])
E.T @ E / len(E)        # [[1, -1], [-1, 1]]: variance 1 each, perfectly anti-correlated
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def fit_kalman(Z, Z_next, Z_obs, Y_obs):
    A = least_squares(____, ____).T
    E = Z_next - Z @ ____.T
    Q = E.T @ E / len(____)
    H = least_squares(____, ____).T
    V = Y_obs - Z_obs @ ____.T
    R = V.T @ V / len(____)
    return ref.stabilise(A), Q, H, R
```

</details>

In [ ]:
# ✅ Check Task 5 (on recording 0)
m0 = rec == 0
fit0 = train & pose_valid & m0
pose_c = pose - pose[fit0].mean(axis=0); emg_c = emg - emg[fit0].mean(axis=0)
Zc, _, Zc_next = make_triples(pose_c, emg_c, rec, fit0)
mine = fit_kalman(Zc, Zc_next, pose_c[fit0], emg_c[fit0])
theirs = ref.fit_kalman(Zc, Zc_next, pose_c[fit0], emg_c[fit0])
for name, a, r_ in zip("AQHR", mine, theirs):
    assert a.shape == r_.shape, f"{name} shape {a.shape}, expected {r_.shape}"
    assert np.allclose(a, r_), f"{name} differs from the reference"
print(f"✅ Kalman model fitted. Q (unpredictable movement per step) averages {np.sqrt(np.diag(mine[1])).mean():.2f}° per joint")

✍️ **Q5.** `Q` is the dynamics noise and `R` the EMG noise. Look at `np.trace(Q)` and `np.trace(R)` (or their diagonals). What does the size of `Q` per joint tell you about how predictable the pose is from one frame to the next? Could some of it be tracking jitter rather than real movement?

> *Your answer:*

## Task 6 · The filter itself
Write the predict–update loop. Start from `z0` and `P0`: before seeing any EMG, the best guess is the mean pose
(`z0 = 0`, since the data are centred) and the uncertainty is the spread of poses in training (`P0` = their covariance).

Per frame `t`, in this order:

```
z  = A z                      # predict
P  = A P Aᵀ + Q
S  = H P Hᵀ + R               # how uncertain the expected EMG is (16 × 16)
K  = P Hᵀ S⁻¹                 # Kalman gain (20 × 16)
z  = z + K (y[t] − H z)       # update with this frame's EMG
P  = (I − K H) P
```

Save `z` and `trace(P)` (the sum of the diagonal: total uncertainty) after each update. For `S⁻¹`, `np.linalg.inv`
is fine here (S is only 16 × 16); `np.linalg.solve` is the more careful choice.

In [ ]:
def kalman_filter(Y: np.ndarray, A: np.ndarray, Q: np.ndarray, H: np.ndarray, R: np.ndarray,
                  z0: np.ndarray, P0: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Decode the hidden pose from EMG, one frame at a time, using only the past.

    Parameters
    ----------
    Y      : (T, 16) float, centred EMG, consecutive frames of ONE recording
    A, Q   : (20, 20); H : (16, 20); R : (16, 16), from fit_kalman
    z0     : (20,) starting estimate; P0 : (20, 20) starting uncertainty

    Returns
    -------
    Z_hat   : (T, 20) float, centred pose estimate after each frame's update
    P_trace : (T,) float, np.trace(P) after each update (total uncertainty)
    """
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 6? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. Copy `z0` and `P0` into `z` and `P`; make empty outputs; the identity `I` (20 × 20).
2. Loop over frames: the six lines in the box above, in that order; store `z` and `np.trace(P)`.
3. The rows-vs-columns warning doesn't apply here: inside the loop `z` is a single (20,) vector, so the formulas
   read exactly as in the maths (`A @ z`, `H @ z`).

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.eye(n)` → identity. `np.trace(P)` → sum of the diagonal. `np.linalg.inv(S)` → S⁻¹.
- For one vector, `M @ z` is "matrix times column vector", just like the maths.

```python
P = np.diag([4.0, 1.0]); H = np.array([[1.0, 0.0]]); R = np.array([[1.0]])
K = P @ H.T @ np.linalg.inv(H @ P @ H.T + R)   # [[0.8], [0.]]: trust the measurement 80% for joint 0,
                                               # and learn nothing about joint 1, which H doesn't see
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def kalman_filter(Y, A, Q, H, R, z0, P0):
    z, P = z0.copy(), P0.copy()
    I = np.eye(len(z0))
    Z_hat = np.empty((len(Y), len(z0))); P_trace = np.empty(len(Y))
    for t in range(len(Y)):
        z = A @ ____
        P = A @ P @ A.T + ____
        S = H @ P @ H.T + ____
        K = P @ H.T @ np.linalg.inv(____)
        z = z + K @ (Y[t] - H @ ____)
        P = (I - K @ ____) @ P
        Z_hat[t], P_trace[t] = z, np.trace(P)
    return Z_hat, P_trace
```

</details>

In [ ]:
# ✅ Check Task 6 (decode the test part of recording 0)
A0, Q0, H0, R0 = theirs
P0_toy = np.cov(pose_c[fit0].T, bias=True)
dec0 = test & m0
Z_hat0, P_tr0 = kalman_filter(emg_c[dec0], A0, Q0, H0, R0, np.zeros(20), P0_toy)
r_Z_hat0, r_P_tr0 = ref.kalman_filter(emg_c[dec0], A0, Q0, H0, R0, np.zeros(20), P0_toy)
assert Z_hat0.shape == (dec0.sum(), 20) and P_tr0.shape == (dec0.sum(),)
assert np.allclose(Z_hat0, r_Z_hat0, atol=1e-6), "decoded poses differ: check the order of the six lines"
assert np.allclose(P_tr0, r_P_tr0, atol=1e-6), "uncertainty differs: check the P updates"
print(f"✅ Kalman filter works. Uncertainty (trace P): {P_tr0[0]:.0f} after the first frame → {P_tr0[-1]:.0f} at the end")

✍️ **Q6.** The filter's uncertainty `trace(P)` settles at a steady level. What two things balance to set that level? What would happen to it, and to the Kalman gain, if the EMG were much noisier (`R` ten times larger)?

> *Your answer:*

## Task 7 · Decode every recording and compare
For each recording: fit on its training frames (with valid pose), decode its **whole** test block from EMG alone,
and score only the test frames with valid pose. (The filter doesn't need the pose to run, so it carries on through
tracking dropouts: only the *scoring* skips them.) Skip recordings with fewer than 50 valid test frames.

Compare three decoders, each calibrated on the same recording:
- **Median pose:** always guess the recording's median training pose. Anything useful must beat this.
- **Static least squares:** `pose ≈ [EMG, 1] Θ`, frame by frame, no memory (your `least_squares` + `add_bias`).
- **Kalman filter:** your `fit_kalman` + `kalman_filter`; add the pose mean back before scoring.

Report the mean over recordings of each decoder's mean angular error (°).

In [ ]:
def angular_error(Y: np.ndarray, Y_hat: np.ndarray) -> float:
    """Mean absolute joint-angle error in degrees over all frames and joints. Y, Y_hat: (T, 20)."""
    return float(np.mean(np.abs(Y - Y_hat)))


decode_err: dict[str, list[float]] = {"median pose": [], "static least squares": [], "Kalman": []}
decoded = np.full_like(pose, np.nan)     # (N, 20): Kalman pose estimate (degrees) on test frames, for the plot
uncertainty = np.full(len(pose), np.nan) # (N,): trace of P on test frames

for r in np.unique(rec):
    fit_r = train & pose_valid & (rec == r)    # fit on these frames
    dec_r = test & (rec == r)                  # decode all of these (the EMG is fine even where pose failed)
    score_r = dec_r & pose_valid               # score only these
    if score_r.sum() < 50:
        continue
    # TODO: median pose → append angular_error(pose[score_r], median) to decode_err["median pose"]
    # TODO: static least squares on add_bias(emg[fit_r]) → pose[fit_r]; predict on emg[score_r]; append the error
    # TODO: centre pose and emg with this recording's training means; make_triples on the centred data with fit_r;
    #       fit_kalman; P0 = np.cov(centred pose[fit_r].T, bias=True); kalman_filter on centred emg[dec_r]
    # TODO: add the pose mean back; store it in decoded[dec_r] and P_trace in uncertainty[dec_r];
    #       append angular_error(pose[score_r], decoded[score_r]) to decode_err["Kalman"]
    ...

for name, errs in decode_err.items():
    print(f"{name:22s} {np.mean(errs):5.2f}°  (over {len(errs)} recordings)")

#### 💡 Stuck on Task 7? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. Median: `np.median(pose[fit_r], axis=0)` is a (20,) guess; broadcasting compares it with every scored frame.
2. Static: `Θ = least_squares(add_bias(emg[fit_r]), pose[fit_r])`, then `add_bias(emg[score_r]) @ Θ`.
3. Kalman: centre with `pose[fit_r].mean(axis=0)` and `emg[fit_r].mean(axis=0)` (training means only, never test),
   fit, filter `emg_c[dec_r]`, add the pose mean back, store, score on `score_r`.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `decoded[dec_r] = Z_hat + mu_z` writes the decoded block back into the full-length array; afterwards
  `decoded[score_r]` picks out just the scored frames, in the right order.

```python
full = np.full(5, np.nan); m = np.array([False, True, True, True, False])
full[m] = [7.0, 8.0, 9.0]; full[m & np.array([1, 0, 1, 1, 1], bool)]   # array([8., 9.])
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
med = np.median(pose[fit_r], axis=0)
    decode_err["median pose"].append(angular_error(pose[score_r], ____))

    Theta = least_squares(add_bias(emg[____]), pose[____])
    decode_err["static least squares"].append(angular_error(pose[score_r], add_bias(emg[____]) @ Theta))

    mu_z, mu_y = pose[fit_r].mean(axis=0), emg[fit_r].mean(axis=0)
    pc, ec = pose - mu_z, emg - mu_y
    Zr, _, Zr_next = make_triples(pc, ec, rec, ____)
    A_r, Q_r, H_r, R_r = fit_kalman(Zr, Zr_next, pc[fit_r], ec[fit_r])
    P0_r = np.cov(pc[fit_r].T, bias=True)
    Z_hat, P_tr = kalman_filter(ec[____], A_r, Q_r, H_r, R_r, np.zeros(20), P0_r)
    decoded[dec_r] = Z_hat + ____
    uncertainty[dec_r] = P_tr
    decode_err["Kalman"].append(angular_error(pose[score_r], decoded[____]))
```

</details>

In [ ]:
# ✅ Check Task 7 (recomputes the reference, about 2 s)
ref_err: dict[str, list[float]] = {k: [] for k in decode_err}
for r in np.unique(rec):
    fit_r, dec_r = train & pose_valid & (rec == r), test & (rec == r)
    score_r = dec_r & pose_valid
    if score_r.sum() < 50:
        continue
    ref_err["median pose"].append(ref.angular_error(pose[score_r], np.median(pose[fit_r], axis=0)))
    Th = ref.least_squares(ref.add_bias(emg[fit_r]), pose[fit_r])
    ref_err["static least squares"].append(ref.angular_error(pose[score_r], ref.add_bias(emg[score_r]) @ Th))
    mu_z, mu_y = pose[fit_r].mean(axis=0), emg[fit_r].mean(axis=0)
    pc, ec = pose - mu_z, emg - mu_y
    Zr, _, Zrn = ref.make_triples(pc, ec, rec, fit_r)
    km = ref.fit_kalman(Zr, Zrn, pc[fit_r], ec[fit_r])
    Zh, _ = ref.kalman_filter(ec[dec_r], *km, np.zeros(20), np.cov(pc[fit_r].T, bias=True))
    ref_err["Kalman"].append(ref.angular_error(pose[score_r], (Zh + mu_z)[pose_valid[dec_r]]))
for name in ref_err:
    assert len(decode_err[name]) == len(ref_err[name]), f"{name}: {len(decode_err[name])} recordings, expected {len(ref_err[name])}"
    assert np.allclose(decode_err[name], ref_err[name], atol=1e-4), f"{name}: errors differ from the reference"
print("✅ All three decoders match the reference")

### 📊 What the Kalman filter decodes
Top: true (black) and decoded (colour) angles for three joints over one recording's test block. Gaps in the black
line are tracking dropouts; the decoder carries on through them. Bottom: the filter's uncertainty `trace(P)`. It
starts high (the decoder knows nothing yet), falls over the first few tenths of a second as EMG comes in, then
sits at a steady level, where the uncertainty added by each predict step balances what each update removes.

In [ ]:
r_plot = 6                                  # a long recording; try others
m = test & (rec == r_plot)
t_s = np.arange(m.sum()) / 50
joints = [5, 9, 13]                         # INDEX_, MIDDLE_, RING_MCP_FE
fig, ax = plt.subplots(2, 1, figsize=(14, 6), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
for i, j in enumerate(joints):
    ax[0].plot(t_s, np.where(pose_valid[m], pose[m, j], np.nan), color="k", lw=1)
    ax[0].plot(t_s, decoded[m, j], color=f"C{i}", lw=1.2, label=JOINT_NAMES[j])
ax[0].set_ylabel("angle (°)"); ax[0].legend(loc="upper right")
ax[0].set_title(f"Recording {r_plot} ({stage_of_rec[r_plot]}): true (black) vs Kalman-decoded")
ax[1].plot(t_s, uncertainty[m], color="C3"); ax[1].set_yscale("log")
ax[1].set_ylabel("trace(P)"); ax[1].set_xlabel("time into the test block (s)")
plt.tight_layout(); plt.show()

✍️ **Q7.** The static map and the Kalman filter both see the same EMG. Where, in the plot of decoded angles, does the Kalman filter help most: during holds or during fast movements? Why?

> *Your answer:*

## ➕ Extra · Put velocity in the state
Wu et al. didn't use position alone: their state held position, **velocity** and acceleration. EMG reflects muscle
*force*, which relates to how the hand is moving as much as where it is, so a state `[pose, velocity]` (40 numbers)
might let the observation model `H` explain the EMG better.

Build the state as `[pose_c(t), pose_c(t) − pose_c(t−1)]`. A frame can only be used for fitting if it **and the frame
before it** are usable training frames of the same recording (otherwise the velocity is fake). Everything else is
your existing code with 40 instead of 20. Score only the first 20 columns (the pose).

In [ ]:
decode_err["Kalman + velocity"] = []
for r in np.unique(rec):
    fit_r, dec_r = train & pose_valid & (rec == r), test & (rec == r)
    score_r = dec_r & pose_valid
    if score_r.sum() < 50:
        continue
    # TODO: centre with this recording's training means (as in Task 7)
    # TODO: vel = differences of the centred pose (first row 0); state = np.hstack([pose_c, vel]) → (N, 40)
    # TODO: fit_v = fit_r with the previous frame also in fit_r (and the same recording)
    # TODO: make_triples / fit_kalman / kalman_filter on the 40-D state; score the first 20 columns + the mean
    ...
print(f"Kalman + velocity: {np.mean(decode_err['Kalman + velocity']):5.2f}°")

#### 💡 Stuck on the extra? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. Velocity at frame t = pose at t minus pose at t−1. `np.diff` gives N−1 rows; put a row of zeros on top so it
   lines up with the frames again.
2. The velocity at t is only real if frame t−1 is usable too: `fit_v[t] = fit_r[t] & fit_r[t−1] & same recording`.
3. Fit and filter exactly as in Task 7 with the 40-column state, `z0 = np.zeros(40)` and a 40 × 40 `P0`.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.r_[np.zeros((1, 20)), np.diff(X, axis=0)]` → (N, 20) differences, first row 0.
- `np.r_[False, fit_r[1:] & fit_r[:-1]]` → True where a frame and its predecessor are both usable.

```python
np.r_[np.zeros((1, 1)), np.diff(np.array([[1.0], [4.0], [6.0]]), axis=0)].ravel()   # [0., 3., 2.]
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
mu_z, mu_y = pose[fit_r].mean(axis=0), emg[fit_r].mean(axis=0)
    pc, ec = pose - mu_z, emg - mu_y
    vel = np.r_[np.zeros((1, 20)), np.diff(____, axis=0)]
    state = np.hstack([pc, ____])
    fit_v = np.r_[False, fit_r[1:] & fit_r[:-1]]
    Zr, _, Zr_next = make_triples(state, ec, rec, ____)
    km = fit_kalman(Zr, Zr_next, state[fit_v], ec[fit_v])
    Z_hat, _ = kalman_filter(ec[dec_r], *km, np.zeros(40), np.cov(state[____].T, bias=True))
    pred = Z_hat[:, :20] + mu_z
    decode_err["Kalman + velocity"].append(angular_error(pose[score_r], pred[pose_valid[dec_r]]))
```

</details>

✍️ **Q8.** Rank the decoders. Does the Kalman filter beat the static map? Does velocity help, and why might it?
What does it mean that *one* decoder for all recordings barely beats the average pose, while per-recording
decoders do better?

> *Your answer:*

### 📊 How sure are we? Confidence intervals over independent blocks
The averages above treat the 29 scored recordings as independent. They are not: the recordings come in **blocks**
of two made **at the same time** (one per hand; the recording number in the file name pairs them). Usually both
hands do the same stage, but in two blocks one hand does `OneHandedFreeStyle` while the other is `unconstrained`
(idle). So the honest unit is the **block** (15 of them). This cell averages each decoder's error per block, then
uses `protocol.session_bootstrap_ci` to get a 95% interval for each decoder and for the **paired difference**
against the median pose. A difference whose interval excludes 0 is a real improvement *for this person*.

In [ ]:
# Ready-made: nothing to fill in. Uses the same recordings, in the same order, as Tasks 7 and the extra.
used = [r for r in np.unique(rec) if (test & pose_valid & (rec == r)).sum() >= 50]
sess_of_used = protocol.session_of_rec(stage_of_rec, filename=filename)[used]   # block id
per_session = {name: np.array([np.mean(np.array(errs)[sess_of_used == s]) for s in np.unique(sess_of_used)])
               for name, errs in decode_err.items()}
print(f"{'decoder':22s} {'mean (95% CI) over 15 blocks':>32s}   {'minus median pose':>26s}")
for name, v in per_session.items():
    m, lo, hi = protocol.session_bootstrap_ci(v)
    dm, dlo, dhi = protocol.session_bootstrap_ci(v - per_session["median pose"])
    print(f"{name:22s} {m:8.2f}° [{lo:5.2f}, {hi:5.2f}]          {dm:+6.2f}° [{dlo:+5.2f}, {dhi:+5.2f}]")

✍️ **Q9.** Which decoders beat the per-recording median pose with an interval that excludes 0? How does using 15 independent blocks instead of 29 recordings change how sure you can be?

> *Your answer:*

---
# Part C · A physical (grey-box) model: activation dynamics + mechanics
**The idea in one picture.** Movement is a loop: brain → muscles → body → senses → brain. We record two points on it:
the **muscles** (EMG) and the **body** (joint angles). So decoding EMG → pose means learning the part of the loop
between them, the body's **forward model** (Almani et al. 2026):

```
EMG ──► activation dynamics ──► a(t) ──► mechanics ──────────────► q(t) ──► 20 joint angles
        (a delay + smoothing:           (force pulls the hand toward a        (eigenposture
         muscles respond slowly)         posture; it accelerates, then         scores → angles)
                                         velocity and position follow)
```

Muscle force sets the **acceleration**; position is the result of force over time. A model with that physics built in
is a **physical** (or "grey-box") model: part physics, part fitted from data. It has far fewer free numbers than Part
B's Kalman + velocity decoder, which could help when generalising.

*Naming note:* "compositional" models (tasks reusing shared building blocks, Lazzari & Saxena 2025) are a different
idea; they are in notebook 07b.

---
## Task 8 · Activation dynamics: muscles respond late and slowly
Two facts from muscle physiology:
1. **Delay.** Force appears tens of milliseconds after the electrical activity: the *electromechanical delay*,
   plus the time for force to move a finger.
2. **Smoothing.** Each twitch lasts much longer than the spike that caused it, so force is a smoothed version of
   the activity (the "fused tetanus" we discussed in notebook 01).

The simplest model of both: delay the input by `delay` frames, then pass it through a **first-order low-pass** with
time constant `τ`, the classic model of muscle activation dynamics:

$$a(t) = a(t-1) + \alpha\,\big(u(t - \text{delay}) - a(t-1)\big), \qquad \alpha = 1 - e^{-\Delta t/\tau},\ \Delta t = 1/50\ \text{s}$$

Each frame `a` moves a fraction `α` of the way toward the (delayed) input. A small `τ` means `α` near 1: `a` follows
the input closely. A large `τ` means `a` changes slowly. It is **causal** (only the past), so it could run live.
Filter **each recording separately**, and start `a` at the recording's first value.

In [ ]:
def activation_filter(U_syn: np.ndarray, rec: np.ndarray, tau_s: float, delay: int, fs: int = 50) -> np.ndarray:
    """Causal first-order low-pass with a pure delay, within each recording.

    Parameters
    ----------
    U_syn     : (N, k) float, input (log activations)
    rec   : (N,) int, recording of each frame
    tau_s : float, time constant in seconds
    delay : int, delay in frames (0 = none)

    Returns
    -------
    A : (N, k) float. In each recording: u is delayed by `delay` frames (the first value repeats to fill the
        start), then a(t) = a(t-1) + alpha * (u_delayed(t) - a(t-1)) with alpha = 1 - exp(-1 / (fs * tau_s)),
        starting from a = u_delayed(first frame).
    """
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 8? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. Compute `alpha` once.
2. Loop over recordings: `idx = np.flatnonzero(rec == r)` gives the rows of one recording, in time order.
3. Delay: put `delay` copies of the first row on top of `u` and drop the last `delay` rows.
4. Loop over time, update `a` with the formula, write each `a` into `out[idx[t]]`.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.repeat(u[:1], 3, axis=0)` → 3 copies of the first row. `np.vstack([top, u[:-3]])` stacks them on top.
- `np.exp(-1 / (50 * 0.1))` → 0.8187, so `alpha` = 0.18 for τ = 0.1 s.

```python
u = np.array([[0.0], [1.0], [1.0], [1.0]]); a = u[0].copy(); alpha = 0.5
for t in range(4):
    a = a + alpha * (u[t] - a); print(a)      # [0.] [0.5] [0.75] [0.875]: rising toward 1
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def activation_filter(U_syn, rec, tau_s, delay, fs=50):
    alpha = 1 - np.exp(-1 / (fs * ____))
    out = np.empty_like(U_syn)
    for r in np.unique(rec):
        idx = np.flatnonzero(rec == r)
        u = U_syn[idx]
        if delay > 0:
            u = np.vstack([np.repeat(u[:1], ____, axis=0), u[:-____]])
        a = u[0].copy()
        for t in range(len(u)):
            a = a + alpha * (____ - a)
            out[idx[t]] = ____
    return out
```

</details>

In [ ]:
# ✅ Check Task 8
few = rec < 3
mine = activation_filter(U_syn[few], rec[few], 0.1, 5)
assert mine.shape == U_syn[few].shape, f"shape {mine.shape}, expected {U_syn[few].shape}"
assert np.allclose(mine, ref.activation_filter(U_syn[few], rec[few], 0.1, 5)), \
    "values differ: check alpha, the delay padding and the starting value"
step = np.r_[np.zeros(10), np.ones(90)][:, None]
resp = activation_filter(step, np.zeros(100, int), 0.1, 0)
assert abs(resp[10 + 5, 0] - (1 - np.exp(-6 / 5))) < 1e-9, "a step should rise as 1 - exp(-t / tau)"
print("✅ Activation filter works. After one τ (0.1 s = 5 frames) a step has risen to "
      f"{resp[14, 0]:.0%} of its final value")

### Task 8b · Choose τ and the delay on training data
Which `τ` and `delay` make the activations most informative about the pose? Score each pair the simple way: in every
recording, fit a straight-line map from the filtered activations to the 20 angles on the first 75% of its
**training** block, and measure the error on the last 25% (`ref.static_error`, ready-made). The **test** block is never
touched, so the choice can't peek at the answer.

In [ ]:
fit_in, val_in = ref.inner_split(train, rec)           # (N,) bool: first 75% / last 25% of each training block
fit_in, val_in = fit_in & pose_valid, val_in & pose_valid
TAUS = [0.02, 0.05, 0.1, 0.2, 0.4]                     # seconds
DELAYS = [0, 2, 5, 10]                                 # frames (0, 40, 100, 200 ms)

grid: dict[tuple[float, int], float] = {}              # (tau, delay) → validation error (°)
# TODO: for every tau and delay: grid[(tau, delay)] = ref.static_error(activation_filter(U_syn, rec, tau, delay),
#                                                                         pose, rec, fit_in, val_in)
# TODO: best = the (tau, delay) with the smallest error;  TAU, DELAY = best
best: tuple[float, int] | None = None

#### 💡 Stuck on Task 8b? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. Two nested loops over `TAUS` and `DELAYS`; store each score in `grid`.
2. The best pair is the key with the smallest value. (About 10 s: 20 filters over 105,000 frames.)

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `min(grid, key=grid.get)` → the key whose value is smallest.

```python
g = {(0.1, 2): 14.6, (0.2, 5): 14.4}
min(g, key=g.get)            # (0.2, 5)
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
for tau in TAUS:
    for dl in DELAYS:
        grid[(tau, dl)] = ref.static_error(activation_filter(U_syn, rec, ____, ____), pose, rec, fit_in, val_in)
best = min(grid, key=____)
TAU, DELAY = best
```

</details>

In [ ]:
# ✅ Check Task 8b
assert len(grid) == 20, f"{len(grid)} pairs, expected 20"
r_best = ref.static_error(ref.activation_filter(U_syn, rec, *best), pose, rec, fit_in, val_in)
assert abs(grid[best] - r_best) < 1e-6, "your score for the best pair differs from the reference"
assert best == min(grid, key=grid.get)
raw = ref.static_error(U_syn, pose, rec, fit_in, val_in)
print(f"✅ Best τ = {best[0]:.2f} s, delay = {best[1]} frames ({best[1] * 20} ms): "
      f"{grid[best]:.2f}° vs {raw:.2f}° with no filtering")
A_act = activation_filter(U_syn, rec, *best)              # (N, 4): the filtered activations used from here on

### 📊 The grid, and what the filter does
Left: validation error for each pair (darker = better). **What to look for:** is there a clear best region, or is the
surface flat? Right: one synergy's raw log activation (grey) and the filtered, delayed version (colour).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 3.5), gridspec_kw={"width_ratios": [1, 2]})
G = np.array([[grid[(t, dl)] for dl in DELAYS] for t in TAUS])
im = ax[0].imshow(G, cmap="viridis_r", aspect="auto")
ax[0].set_xticks(range(len(DELAYS)), [f"{dl * 20} ms" for dl in DELAYS]); ax[0].set_xlabel("delay")
ax[0].set_yticks(range(len(TAUS)), [f"{t:.2f} s" for t in TAUS]); ax[0].set_ylabel("τ")
plt.colorbar(im, ax=ax[0], label="validation error (°)")
m = rec == 6; t_s = np.arange(m.sum()) / 50; sl = slice(0, 500)
ax[1].plot(t_s[sl], U_syn[m][sl, 0], color="0.6", lw=0.8, label="raw log activation")
ax[1].plot(t_s[sl], A_act[m][sl, 0], color="C0", lw=1.5, label=f"τ = {TAU} s, delay = {DELAY * 20} ms")
ax[1].set_xlabel("time (s)"); ax[1].legend(); ax[1].set_title("synergy 1, recording 6")
plt.tight_layout(); plt.show()

✍️ **Q10.** Is the delay you found plausible as an electromechanical delay plus the time a finger takes to start moving? The improvement over no filtering is real but small: what does that say about how much of the pose an *instantaneous* map can ever get from the activations?

> *Your answer:*

---
## Task 9 · Mechanics: a spring-damper hand pulled toward a commanded posture
**The model.** Treat each eigenposture score `q_i` as a mass on a spring. The synergies set where the spring pulls
to (an *equilibrium posture*, Feldman's idea: the brain sets a target, the muscles' spring-like stiffness does the
rest), and a damper resists speed:

$$\ddot q = -K\,(q - G\,a) - D\,\dot q + b \;=\; -K q - D \dot q + M a + b, \qquad M = K G$$

`K` (stiffness) and `D` (damping) are **diagonal**: one spring per eigenposture. That is a strong simplification
(the eigenpostures are uncorrelated on average, not mechanically independent), and it keeps the model tiny:
2 numbers per dimension plus the input weights `M` (n × k).

**One frame at a time** (Δt = 1 frame, velocity `v` in "per frame"):

    v(t+1) = v(t) − K q(t) − D v(t) + M a(t) + b
    q(t+1) = q(t) + v(t+1)

**Stability.** With the input off, the state `[q, v]` evolves by a 2n × 2n matrix `F`. Each eigenposture gives a 2 × 2
block `[[1−K, 1−D], [−K, 1−D]]`, whose eigenvalues have product `1 − D` and sum `2 − K − D`. Working through the
conditions for both eigenvalues to lie inside the unit circle gives **`D > 0` and `0 < K < 4 − 2D`**. So
**positive stiffness and damping (and not too large) guarantee that every free motion dies out.** That is why Task 10
constrains them to be non-negative: it builds in a fact about real hands.

In [ ]:
def transition_matrix(K: np.ndarray, D: np.ndarray) -> np.ndarray:
    """(2n, 2n) transition of the state [q, v] with zero input.

    Rows 0…n-1 give q(t+1), rows n…2n-1 give v(t+1). From the update above:
    q(t+1) = (1-K) q + (1-D) v,   v(t+1) = -K q + (1-D) v   (each a diagonal block).
    """
    # TODO
    raise NotImplementedError


def simulate(q0: np.ndarray, v0: np.ndarray, A_in: np.ndarray, K: np.ndarray, D: np.ndarray,
             M: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Run the model forward on its own predictions.

    Parameters
    ----------
    q0, v0 : (n,) starting position and velocity (v0 = q(t0) - q(t0 - 1))
    A_in   : (H, k) input a(t0), a(t0 + 1), …, one row per step
    K, D, b: (n,);  M : (n, k)

    Returns
    -------
    (H, n): q after 1, 2, …, H frames, using the two update lines above.
    """
    # TODO
    raise NotImplementedError


def max_abs_eig(K: np.ndarray, D: np.ndarray) -> float:
    """Largest |eigenvalue| of transition_matrix(K, D). Below 1 = stable."""
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 9? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. `transition_matrix`: start from zeros (2n × 2n) and fill the four n × n blocks with diagonal matrices.
2. `simulate`: copy `q0` and `v0`, loop over the rows of `A_in`, update `v` first, then `q` with the **new** `v`,
   store `q`.
3. `max_abs_eig`: eigenvalues of the matrix, absolute value, maximum.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.diag(x)` turns a vector into a diagonal matrix. `F[:n, n:] = …` fills the top-right block.
- `K * q` multiplies element by element (diagonal K); `M @ a` is a matrix-vector product.
- `np.linalg.eigvals(F)` → complex eigenvalues; `np.abs(...)` → their sizes.

```python
F = np.zeros((4, 4)); F[:2, 2:] = np.diag([1.0, 2.0])   # top-right block
np.abs(np.linalg.eigvals(np.array([[0.0, 1.0], [-0.81, 0.0]]))).max()   # 0.9: a decaying oscillation
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def transition_matrix(K, D):
    n = len(K); F = np.zeros((2 * n, 2 * n))
    F[:n, :n] = np.diag(1 - ____); F[:n, n:] = np.diag(1 - ____)
    F[n:, :n] = np.diag(-____);    F[n:, n:] = np.diag(1 - ____)
    return F

def simulate(q0, v0, A_in, K, D, M, b):
    q, v = q0.astype(float).copy(), v0.astype(float).copy()
    out = np.empty((len(A_in), len(q0)))
    for h in range(len(A_in)):
        v = v - K * q - D * v + M @ ____ + b
        q = q + ____
        out[h] = q
    return out

def max_abs_eig(K, D):
    return float(np.abs(np.linalg.eigvals(transition_matrix(____, ____))).max())
```

</details>

In [ ]:
# ✅ Check Task 9
rng = np.random.default_rng(1)
K_t, D_t = np.array([0.02, 0.05, 0.1]), np.array([0.1, 0.2, 0.3])
M_t, b_t = rng.normal(0, 0.05, (3, 2)), np.array([0.01, 0.0, -0.01])
A_t = rng.normal(size=(40, 2))
assert np.allclose(transition_matrix(K_t, D_t), ref.transition_matrix(K_t, D_t)), "transition matrix differs"
sim = simulate(np.ones(3), np.zeros(3), A_t, K_t, D_t, M_t, b_t)
assert sim.shape == (40, 3)
assert np.allclose(sim, ref.simulate(np.ones(3), np.zeros(3), A_t, K_t, D_t, M_t, b_t)), \
    "simulate differs: update v first, then q with the new v"
assert abs(max_abs_eig(K_t, D_t) - ref.max_abs_eig(K_t, D_t)) < 1e-12
assert max_abs_eig(np.array([0.1]), np.array([-0.05])) > 1, "negative damping should be unstable"
print(f"✅ Mechanics work. This test system's largest |eigenvalue| = {max_abs_eig(K_t, D_t):.3f} (stable)")

### 📊 What K and D do
A single eigenposture released from `q = 1` with no input, for three settings. **What to look for:** low damping
oscillates for a long time (eigenvalues near 1 in size); high damping creeps back without overshoot; and the
stiffness sets how fast it returns.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
for K1, D1, lab in [(0.05, 0.05, "soft spring, little damping"), (0.05, 0.4, "soft spring, heavy damping"),
                    (0.3, 0.3, "stiff spring")]:
    x = simulate(np.ones(1), np.zeros(1), np.zeros((100, 1)), np.array([K1]), np.array([D1]), np.zeros((1, 1)),
                 np.zeros(1))
    ax.plot(np.arange(1, 101) * 20, x[:, 0], label=f"{lab}: K={K1}, D={D1}, max|eig|={max_abs_eig(np.array([K1]), np.array([D1])):.3f}")
ax.axhline(0, color="k", lw=0.5); ax.set_xlabel("time (ms)"); ax.set_ylabel("q"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

✍️ **Q11.** Using the 2 × 2 block, explain why `D = 0` (no damping) can never be stable, whatever `K` is. What would an unstable fitted model do in a long rollout, and why is that a reason to *constrain* the fit rather than hope the data gets it right?

> *Your answer:*

---
## Task 10 · Fit the spring-damper by least squares on finite differences
We never measure velocity or acceleration; we estimate them from consecutive frames:

    v(t)   = q(t) − q(t−1)
    acc(t) = q(t+1) − 2 q(t) + q(t−1)        (= v(t+1) − v(t))

So a frame `t` is usable only if `t−1`, `t` and `t+1` are all good frames **of the same recording** (Task 1 of Part A,
one step longer). Then, for each eigenposture `i` separately, the model is a linear regression:

    acc_i = −K_i q_i − D_i v_i + M_i · a + b_i        regressors [q_i, v_i, a_1 … a_k, 1]

**The constraint (an "active set").** Fit freely. If `K_i` or `D_i` comes out **negative** (unphysical, and possibly
unstable), fix it at 0 and refit the remaining coefficients. Repeat at most a few times. A tiny ridge term
(`1e-6 · I`) keeps the solve well behaved.

First you'll check the fit on a **simulated hand** whose true K, D and M you know, the standard sanity check before
trusting a fit on real data.

In [ ]:
def second_order_steps(mask: np.ndarray, rec: np.ndarray) -> np.ndarray:
    """(M,) int: frames t where t-1, t and t+1 are all in mask and t-1, t+1 are in the same recording."""
    # TODO
    raise NotImplementedError


def fit_spring_damper(Q: np.ndarray, A: np.ndarray, t: np.ndarray, ridge: float = 1e-6
                      ) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Fit acc_i = -K_i q_i - D_i v_i + M_i · a + b_i for every pose dimension i, with K_i, D_i >= 0.

    Parameters
    ----------
    Q : (N, n) pose scores;  A : (N, k) activations;  t : (M,) frames from second_order_steps

    Returns
    -------
    K (n,), D (n,), M (n, k), b (n,)
    Solve each regression with np.linalg.solve(X.T @ X + ridge * I, X.T @ y). If K_i or D_i is negative, drop that
    regressor (fix it at 0) and solve again; at most 3 rounds.
    """
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 10? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. `second_order_steps`: three shifted copies of `mask` (`mask[:-2]`, `mask[1:-1]`, `mask[2:]`) must all be True, and
   `rec[:-2] == rec[2:]`. The index of the middle frame is `np.flatnonzero(ok) + 1`.
2. `fit_spring_damper`: build `qp, q, qn, a = Q[t-1], Q[t], Q[t+1], A[t]`, then `v` and `acc`.
3. For each `i`: keep a list of which of `[q_i, v_i]` are still free; stack the free ones with `a` and a column of
   ones; solve; read `K_i = −coef(q_i)`, `D_i = −coef(v_i)` (0 if fixed); if any is negative, mark it fixed and solve again.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.column_stack([x1, x2, A, np.ones(len(x1))])` → a design matrix with columns side by side.
- Reading coefficients back in order: keep a counter `j` that moves past each free regressor.

```python
mask = np.array([1, 1, 1, 0, 1, 1, 1], bool); rec = np.zeros(7, int)
ok = mask[:-2] & mask[1:-1] & mask[2:] & (rec[:-2] == rec[2:])
np.flatnonzero(ok) + 1          # array([1, 5]): frames whose both neighbours are usable
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def second_order_steps(mask, rec):
    ok = mask[:-2] & mask[1:-1] & mask[2:] & (rec[:-2] == rec[____])
    return np.flatnonzero(ok) + ____

def fit_spring_damper(Q, A, t, ridge=1e-6):
    qp, q, qn, a = Q[t - 1], Q[t], Q[t + 1], A[t]
    v, acc = q - qp, qn - 2 * q + ____
    n, k = Q.shape[1], A.shape[1]
    K, D, M, b = np.zeros(n), np.zeros(n), np.zeros((n, k)), np.zeros(n)
    for i in range(n):
        free, own = [True, True], [q[:, i], v[:, i]]
        for _ in range(3):
            X = np.column_stack([c for c, f in zip(own, free) if f] + [a, np.ones(len(t))])
            th = np.linalg.solve(X.T @ X + ridge * np.eye(X.shape[1]), X.T @ acc[:, ____])
            kd, j = [0.0, 0.0], 0
            for m in range(2):
                if free[m]:
                    kd[m] = -th[j]; j += 1
            neg = [m for m in range(2) if free[m] and kd[m] < 0]
            if not neg:
                break
            for m in neg:
                free[m] = ____
        K[i], D[i] = kd
        M[i], b[i] = th[j:j + k], th[j + ____]
    return K, D, M, b
```

</details>

In [ ]:
# ✅ Check Task 10 · first on a simulated hand with known K, D, M
rng = np.random.default_rng(0)
T_sim = 20000
a_sim = np.zeros((T_sim, 2))
for i in range(1, T_sim):                       # smooth random input
    a_sim[i] = 0.95 * a_sim[i - 1] + rng.normal(0, 0.3, 2)
q_sim = np.zeros((T_sim, 3)); qq, vv = np.zeros(3), np.zeros(3)
for i in range(T_sim):                          # the true model, plus a little random force
    vv = vv - K_t * qq - D_t * vv + M_t @ a_sim[i] + rng.normal(0, 0.01, 3); qq = qq + vv; q_sim[i] = qq
t_sim = second_order_steps(np.ones(T_sim, bool), np.zeros(T_sim, int))
q_sim = np.r_[np.zeros((1, 3)), q_sim[:-1]]     # align: row t holds q(t), so acc(t) uses a(t)
K_f, D_f, M_f, _ = fit_spring_damper(q_sim, a_sim, t_sim)
assert np.allclose(K_f, K_t, rtol=0.1) and np.allclose(D_f, D_t, rtol=0.1), f"recovered K {K_f}, D {D_f}"
assert np.allclose(M_f, M_t, atol=0.01), "input weights M not recovered"
# … then on real data, against the reference
fit_all = train & pose_valid
t_real = second_order_steps(fit_all, rec)
assert np.array_equal(t_real, ref.second_order_steps(fit_all, rec)), "second_order_steps differs from the reference"
A_c = A_act - A_act[fit_all].mean(axis=0)       # centred activations
fit_mine = fit_spring_damper(Qs, A_c, t_real)
for name, x, y in zip("KDMb", fit_mine, ref.fit_spring_damper(Qs, A_c, t_real)):
    assert np.allclose(x, y, atol=1e-8), f"{name} differs from the reference"
K_r, D_r, M_r, b_r = fit_mine
print(f"✅ Recovered the simulated hand (K {K_f.round(3)}, D {D_f.round(3)}). On real data: "
      f"{len(t_real):,} steps, max |eigenvalue| = {max_abs_eig(K_r, D_r):.3f}")

### 📊 Fitted stiffness and damping, the stored pose vs an extra 5 Hz low-pass
The same fit on the raw eigenposture scores and on scores **low-passed at 5 Hz** (`ref.smooth_per_recording`).
**What to look for:** how much the damping `D` changes. Tracking jitter makes the finite-difference velocity noisy,
and noise in a regressor biases its coefficient (here: makes the hand look far more damped than it is).

In [ ]:
Qs_smooth = ref.smooth_per_recording(Qs, rec)
K_s, D_s, _, _ = fit_spring_damper(Qs_smooth, A_c, t_real)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.2), sharex=True)
x = np.arange(K_POSE)
for a_, vals, lab in [(ax[0], (K_r, K_s), "stiffness K"), (ax[1], (D_r, D_s), "damping D")]:
    a_.bar(x - 0.2, vals[0], 0.4, label="pose as stored (6 Hz)"); a_.bar(x + 0.2, vals[1], 0.4, label="extra low-pass at 5 Hz")
    a_.set_title(lab); a_.set_xlabel("eigenposture"); a_.legend(fontsize=8)
plt.tight_layout(); plt.show()
print(f"mean D: as stored {D_r.mean():.3f}, extra 5 Hz low-pass {D_s.mean():.3f} | "
      f"max|eig|: as stored {max_abs_eig(K_r, D_r):.3f}, extra low-pass {max_abs_eig(K_s, D_s):.3f}")

✍️ **Q12.** Look at the mean damping `D` printed above. A damping of 0.5 per frame would mean half the velocity
disappears every 20 ms; on the *unsmoothed* pose (data version 1) the fit gave about 0.5. Is the value you get on the
stored pose (low-passed at 6 Hz in notebook 01) plausible for a hand? Why does noise in the *velocity regressor* bias
`D` upward (think of what one jittery frame does to v(t) and to acc(t) at the same time)? Does the extra 5 Hz
low-pass still change `D` much?

> *Your answer:*

---
## Task 11 · Is the hand driven by the EMG? Free rollouts
The research plan's H4: *if movement is input-driven, a model with EMG input beats an autonomous model in free
rollouts.* A one-step prediction is too easy a test (the hand barely moves in 20 ms), so the review's advice is to
judge models on **free rollouts**: start from a real pose and run the model forward on its own predictions.

For each recording, fit the spring-damper on its training frames (raw pose), then from test start points
(every 10th frame whose next 50 frames are usable) run three predictions for 50 frames (1 s):
- **driven:** `simulate` with the real filtered activations;
- **input off:** the same model with the input held at its training mean (zeros, since `A_c` is centred);
- **no change:** the pose stays where it started.

Score every prediction against the **projected** pose (`to_angles(Qs)`), because the model lives in 10 eigenposture
dimensions; this way all three are judged in the same space.

In [ ]:
H_MAX, HORIZONS = 50, [1, 5, 25, 50]
Q_proj = to_angles(Qs)                         # (N, 20) the pose as the 10-D model can represent it
roll: dict[str, list[np.ndarray]] = {"no change": [], "input off": [], "driven": []}   # each item: (50,) error per step

for r in np.unique(rec):
    fit_r = train & pose_valid & (rec == r)
    starts = ref.rollout_starts(test & pose_valid & (rec == r), rec, H_MAX + 1)
    starts = starts[starts >= 1][::10]
    if len(starts) == 0:
        continue
    # TODO: centre A_act with fit_r's mean → A_r; fit_spring_damper(Qs, A_r, second_order_steps(fit_r, rec))
    # TODO: for each start s: q0 = Qs[s], v0 = Qs[s] - Qs[s - 1], truth = Q_proj[s + 1 : s + 1 + H_MAX]
    #       driven = to_angles(simulate(q0, v0, A_r[s : s + H_MAX], K, D, M, b))
    #       off    = to_angles(simulate(q0, v0, np.zeros((H_MAX, K_SYN)), K, D, M, b))
    #       none   = Q_proj[s] repeated H_MAX times
    #       append np.abs(prediction - truth).mean(axis=1) for each model to roll[...]
    ...

for name, e in roll.items():
    e = np.mean(e, axis=0)
    print(f"{name:10s} " + "  ".join(f"{h:2d} steps {e[h - 1]:5.2f}°" for h in HORIZONS))

#### 💡 Stuck on Task 11? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. Per recording: centre the activations with that recording's training mean, find its usable steps, fit.
2. Per start: build the starting position and velocity, the 50-frame truth, and the three predictions.
3. The error per step is the mean over the 20 joints: `np.abs(pred - truth).mean(axis=1)` → (50,).

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.tile(row, (50, 1))` repeats a (20,) row 50 times → (50, 20).
- `A_r[s : s + 50]` are the inputs a(s), …, a(s + 49): exactly what `simulate` needs to produce q(s+1) … q(s+50).

```python
truth = np.zeros((3, 2)); pred = np.ones((3, 2))
np.abs(pred - truth).mean(axis=1)          # array([1., 1., 1.]): one error per step
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
A_r = A_act - A_act[fit_r].mean(axis=0)
    K, D, M, b = fit_spring_damper(Qs, A_r, second_order_steps(____, rec))
    for s in starts:
        q0, v0 = Qs[s], Qs[s] - Qs[s - 1]
        truth = Q_proj[s + 1 : s + 1 + H_MAX]
        preds = {"driven": to_angles(simulate(q0, v0, A_r[s : s + ____], K, D, M, b)),
                 "input off": to_angles(simulate(q0, v0, np.zeros((H_MAX, K_SYN)), K, D, M, b)),
                 "no change": np.tile(Q_proj[____], (H_MAX, 1))}
        for name, p in preds.items():
            roll[name].append(np.abs(p - ____).mean(axis=1))
```

</details>

In [ ]:
# ✅ Check Task 11 (recomputes the reference, a few seconds)
r_roll = {k: [] for k in roll}
for r in np.unique(rec):
    fit_r = train & pose_valid & (rec == r)
    st = ref.rollout_starts(test & pose_valid & (rec == r), rec, H_MAX + 1); st = st[st >= 1][::10]
    if len(st) == 0:
        continue
    A_r = A_act - A_act[fit_r].mean(axis=0)
    p = ref.fit_spring_damper(Qs, A_r, ref.second_order_steps(fit_r, rec))
    for s in st:
        tr_ = Q_proj[s + 1: s + 1 + H_MAX]
        r_roll["driven"].append(np.abs(to_angles(ref.simulate(Qs[s], Qs[s] - Qs[s - 1], A_r[s: s + H_MAX], *p)) - tr_).mean(1))
        r_roll["input off"].append(np.abs(to_angles(ref.simulate(Qs[s], Qs[s] - Qs[s - 1], np.zeros((H_MAX, K_SYN)), *p)) - tr_).mean(1))
        r_roll["no change"].append(np.abs(np.tile(Q_proj[s], (H_MAX, 1)) - tr_).mean(1))
for k in roll:
    assert len(roll[k]) == len(r_roll[k]), f"{k}: {len(roll[k])} rollouts, expected {len(r_roll[k])}"
    assert np.allclose(np.mean(roll[k], 0), np.mean(r_roll[k], 0), atol=1e-6), f"{k}: errors differ from the reference"
print(f"✅ Rollouts match the reference ({len(roll['driven'])} starting points)")

### 📊 Driven vs autonomous vs no change
**What to look for:** H4 predicts the **driven** curve sits clearly below **input off**, especially at long horizons
(the EMG should keep steering the hand). If the two curves overlap, the input adds little to this model's forecasts.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for name, e in roll.items():
    ax.plot(np.arange(1, H_MAX + 1) * 20, np.mean(e, axis=0), marker="o", ms=2, label=name)
ax.set_xlabel("time ahead (ms)"); ax.set_ylabel("mean angular error (°)"); ax.legend()
ax.set_title("Free rollouts on test data (projected pose)")
plt.tight_layout(); plt.show()

✍️ **Q13.** Does the EMG input help the rollouts? Part A found the same for an unconstrained linear model. Give two reasons a *forecast* can fail even if the EMG really drives the hand (think about what a 100 ms filtered activation can know about a movement 1 s later, and about errors compounding).

> *Your answer:*

---
## Task 12 · The physical model as a Kalman decoder
A forecast runs blind. A **decoder** corrects itself with every new frame of EMG: that is the Kalman filter from
Part B (the review's equation 9: predict with the forward model, then correct with the "sensory" signal).

**State** `z = [q, v, a]` (10 + 10 + 4 = 24 numbers). **Dynamics** = the physical model:

    q, v:  the spring-damper of Task 9, driven by a          (the mechanics)
    a:     a(t+1) = ρ a(t)  per synergy                      (activations change slowly; ρ from least squares)

which is one big transition matrix `F` (24 × 24):

| block | rows q | rows v | rows a |
|---|---|---|---|
| **from q** | `diag(1−K)` | `diag(−K)` | 0 |
| **from v** | `diag(1−D)` | `diag(1−D)` | 0 |
| **from a** | `M` | `M` | `diag(ρ)` |

(read it as "row block ← column block": `q(t+1) = (1−K) q + (1−D) v + M a`, and so on; in code `F[:n, 2n:] = M`).

**Observation**: the 16-channel log EMG, `y = H z + noise`, with `H` and `R` fitted by least squares as in Part B.
`Q` (how much the state is pushed around each frame) is the covariance of the one-step residuals `z(t+1) − F z(t)`.

Compared with Part B's Kalman + velocity decoder (a free 40 × 40 matrix, 1,600 dynamics numbers), the physical
dynamics have about 70 free numbers. **Fewer free numbers is the whole point:** it should need less data.

In [ ]:
def physical_matrices(K: np.ndarray, D: np.ndarray, M: np.ndarray, rho: np.ndarray) -> np.ndarray:
    """(2n + k, 2n + k) transition of z = [q, v, a]: transition_matrix(K, D) in the top-left, M feeding a into
    both q and v, and diag(rho) for a."""
    # TODO
    raise NotImplementedError


def physical_state(Qc: np.ndarray, Ac: np.ndarray) -> np.ndarray:
    """(N, 2n + k) state [q, v, a] with v(t) = q(t) - q(t-1) and a first row of 0 for v. Inputs already centred."""
    # TODO
    raise NotImplementedError


def fit_physical(Qs: np.ndarray, A: np.ndarray, Y: np.ndarray, fit_mask: np.ndarray, rec: np.ndarray
                  ) -> dict[str, np.ndarray]:
    """Fit every part on fit_mask frames.

    1. t = second_order_steps(fit_mask, rec); centre Qs, A, Y with their fit_mask means (mu_q, mu_a, mu_y)
    2. K, D, M, _ = fit_spring_damper(Qc, Ac, t);  rho_j = sum(a_j(t) a_j(t+1)) / sum(a_j(t)^2) over t
    3. F = physical_matrices(...);  Z = physical_state(Qc, Ac)
    4. Q = covariance of Z[t+1] - Z[t] @ F.T;  H = least_squares(Z[t], Yc[t]).T;  R = covariance of Yc[t] - Z[t] @ H.T
       (covariance here = res.T @ res / len(t), as in Part B);  P0 = np.cov(Z[t].T, bias=True)
    Returns {"F", "Q", "H", "R", "P0", "mu_q", "mu_y"} (you may add more keys).
    """
    # TODO
    raise NotImplementedError


def physical_decode(model: dict[str, np.ndarray], Y_block: np.ndarray) -> np.ndarray:
    """(T, n) decoded pose scores for one contiguous block of observations: ref.kalman_filter on
    Y_block - mu_y from z0 = 0 and P0; keep the first n columns; add mu_q back."""
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 12? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. `physical_matrices`: zeros of size 2n + k; put `transition_matrix(K, D)` top-left; `M` into rows `:n` and rows
   `n:2n`, columns `2n:`; `np.diag(rho)` bottom-right.
2. `physical_state`: velocity = row differences with a zero row on top; stack `[q, v, a]` side by side.
3. `fit_physical`: follow the four numbered steps of the docstring. Everything except `rho` is code you already wrote.
4. `physical_decode`: three lines.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `(a0 * a1).sum(axis=0) / (a0 * a0).sum(axis=0)` is the least-squares slope of `a1 ≈ ρ a0`, one per column.
- `ref.least_squares(X, Y)` solves the normal equations (Part A); `.T` turns (n_state, 16) into H (16, n_state).

```python
a0 = np.array([[1.0], [2.0]]); a1 = np.array([[0.9], [1.8]])
(a0 * a1).sum(0) / (a0 * a0).sum(0)        # array([0.9])
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def physical_matrices(K, D, M, rho):
    n, k = len(K), len(rho); F = np.zeros((2 * n + k, 2 * n + k))
    F[:2 * n, :2 * n] = transition_matrix(K, D)
    F[:n, 2 * n:] = ____; F[n:2 * n, 2 * n:] = ____
    F[2 * n:, 2 * n:] = np.diag(____)
    return F

def physical_state(Qc, Ac):
    v = np.vstack([np.zeros((1, Qc.shape[1])), np.diff(Qc, axis=____)])
    return np.hstack([Qc, v, Ac])

def fit_physical(Qs, A, Y, fit_mask, rec):
    t = second_order_steps(fit_mask, rec)
    mu_q, mu_a, mu_y = Qs[fit_mask].mean(0), A[fit_mask].mean(0), Y[fit_mask].mean(0)
    Qc, Ac, Yc = Qs - mu_q, A - mu_a, Y - mu_y
    K, D, M, _ = fit_spring_damper(Qc, Ac, t)
    rho = (Ac[t] * Ac[t + 1]).sum(0) / (Ac[t] * Ac[t]).sum(0)
    F = physical_matrices(K, D, M, rho)
    Z = physical_state(Qc, Ac)
    res = Z[t + 1] - Z[t] @ F.T
    H = ref.least_squares(Z[t], Yc[____]).T
    rv = Yc[t] - Z[t] @ H.T
    return {"F": F, "Q": res.T @ res / len(t), "H": H, "R": rv.T @ rv / len(t),
            "P0": np.cov(Z[t].T, bias=True), "mu_q": mu_q, "mu_y": mu_y}

def physical_decode(model, Y_block):
    n = len(model["mu_q"])
    Zh, _ = ref.kalman_filter(Y_block - model["mu_y"], model["F"], model["Q"], model["H"], model["R"],
                                np.zeros(len(model["F"])), model["P0"])
    return Zh[:, :____] + model["mu_q"]
```

</details>

In [ ]:
# ✅ Check Task 12 (one recording)
r0 = 6
fit_r, dec_r = train & pose_valid & (rec == r0), test & (rec == r0)
mine, theirs = fit_physical(Qs, A_act, LE, fit_r, rec), ref.fit_physical(Qs, A_act, LE, fit_r, rec)
for key in ["F", "Q", "H", "R", "P0", "mu_q", "mu_y"]:
    assert np.allclose(mine[key], theirs[key], atol=1e-8), f"{key} differs from the reference"
dec_mine = physical_decode(mine, LE[dec_r])
assert dec_mine.shape == (dec_r.sum(), K_POSE)
assert np.allclose(dec_mine, ref.physical_decode(theirs, LE[dec_r]), atol=1e-6), "decoded scores differ"
print(f"✅ Physical decoder works. Largest |eigenvalue| of F = {np.abs(np.linalg.eigvals(mine['F'])).max():.3f}; "
      f"recording {r0} error = {ref.angular_error(pose[dec_r & pose_valid], to_angles(dec_mine)[pose_valid[dec_r]]):.2f}°")

✍️ **Q14.** The physical `F` has about 70 free numbers; Part B's Kalman + velocity has 1,600. Which do you expect to win *within* a recording (plenty of data from the same movement), and which on a stage it has never seen? Write your prediction before Task 13.

> *Your answer:*

---
## Task 13 · Compare decoders within recordings, with a null
Same set-up as Part B: each recording gets its own decoders, fitted on its first 80% and scored on its last 20%.
Five decoders:

| Decoder | Uses EMG? | State / dynamics? |
|---|---|---|
| per-recording **median pose** | no | no |
| **static log-EMG**: `pose ≈ [LE, 1] Θ` | yes | no |
| **Kalman + velocity** (Part B's extra, on log EMG; ready-made `ref.fit_kalman_velocity`) | yes | free 40 × 40 |
| **physical** (yours) | yes | spring-damper + activation dynamics |
| **physical, shifted EMG** (null) | EMG shifted in time | same |

**The null** (`protocol.circular_shift_null`): shift the EMG-derived signals (`LE` and `A_act`, the same shift for both)
circularly within each recording by at least 5 s, and refit everything. The EMG keeps all its own structure, but its
alignment with the pose is gone. Anything the physical decoder does **better than this null** comes from the EMG
actually tracking the hand, not from the dynamics alone.

In [ ]:
rng = np.random.default_rng(SEED)
shifted = protocol.circular_shift_null(np.hstack([LE, A_act]), rec, rng)   # same shift for both, per recording
LE_null, A_null = shifted[:, :16], shifted[:, 16:]

within: dict[str, list[float]] = {k: [] for k in ["median pose", "static log-EMG", "Kalman + velocity",
                                                  "physical", "physical, shifted EMG"]}
used: list[int] = []                                   # recordings scored, in order
for r in np.unique(rec):
    fit_r, dec_r = train & pose_valid & (rec == r), test & (rec == r)
    score_r = dec_r & pose_valid
    if score_r.sum() < 50:
        continue
    used.append(r)
    # TODO: median pose: np.median(pose[fit_r], axis=0) vs pose[score_r]
    # TODO: static log-EMG: ref.least_squares(ref.add_bias(LE[fit_r]), pose[fit_r]), predict on LE[score_r]
    # TODO: Kalman + velocity: m = ref.fit_kalman_velocity(pose, LE, fit_r, rec); ref.kalman_velocity_decode(m, LE[dec_r])
    # TODO: physical: fit_physical(Qs, A_act, LE, fit_r, rec); to_angles(physical_decode(model, LE[dec_r]))
    # TODO: physical, shifted EMG: the same with A_null and LE_null
    # Score each decoded block on the valid frames only: prediction[pose_valid[dec_r]] vs pose[score_r]
    ...

for name, e in within.items():
    print(f"{name:24s} {np.mean(e):5.2f}°  (over {len(e)} recordings)")

#### 💡 Stuck on Task 13? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. Each decoder is one or two lines; append `ref.angular_error(pose[score_r], prediction)` to its list.
2. The Kalman decoders return the **whole** test block (`dec_r`), including frames where tracking failed (the EMG is
   fine there). Score only the valid ones: `block[pose_valid[dec_r]]`.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `pose_valid[dec_r]` is a (T,) mask over the frames of the decoded block, in the same order.

```python
block = np.arange(5)[:, None]; valid_in_block = np.array([1, 0, 1, 1, 0], bool)
block[valid_in_block].ravel()          # array([0, 2, 3])
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
vb = pose_valid[dec_r]
    within["median pose"].append(ref.angular_error(pose[score_r], np.median(pose[fit_r], axis=0)))
    th = ref.least_squares(ref.add_bias(LE[fit_r]), pose[fit_r])
    within["static log-EMG"].append(ref.angular_error(pose[score_r], ref.add_bias(LE[____]) @ th))
    kv = ref.kalman_velocity_decode(ref.fit_kalman_velocity(pose, LE, fit_r, rec), LE[dec_r])
    within["Kalman + velocity"].append(ref.angular_error(pose[score_r], kv[____]))
    comp = to_angles(physical_decode(fit_physical(Qs, A_act, LE, fit_r, rec), LE[dec_r]))
    within["physical"].append(ref.angular_error(pose[score_r], comp[vb]))
    null = to_angles(physical_decode(fit_physical(Qs, ____, ____, fit_r, rec), LE_null[dec_r]))
    within["physical, shifted EMG"].append(ref.angular_error(pose[score_r], null[vb]))
```

</details>

In [ ]:
# ✅ Check Task 13 (two recordings against the reference)
for r in used[:2]:
    i = used.index(r)
    fit_r, dec_r = train & pose_valid & (rec == r), test & (rec == r)
    vb, score_r = pose_valid[dec_r], dec_r & pose_valid
    comp = to_angles(ref.physical_decode(ref.fit_physical(Qs, A_act, LE, fit_r, rec), LE[dec_r]))
    assert abs(within["physical"][i] - ref.angular_error(pose[score_r], comp[vb])) < 1e-6, f"physical, recording {r}"
    null = to_angles(ref.physical_decode(ref.fit_physical(Qs, A_null, LE_null, fit_r, rec), LE_null[dec_r]))
    assert abs(within["physical, shifted EMG"][i] - ref.angular_error(pose[score_r], null[vb])) < 1e-6, "null"
    kv = ref.kalman_velocity_decode(ref.fit_kalman_velocity(pose, LE, fit_r, rec), LE[dec_r])
    assert abs(within["Kalman + velocity"][i] - ref.angular_error(pose[score_r], kv[vb])) < 1e-6, "Kalman + velocity"
assert all(len(v) == len(used) for v in within.values()), "every decoder needs one error per scored recording"
print("✅ Decoders match the reference")

### 📊 How sure are we? CIs over the 15 independent blocks
Recordings come in **blocks** of two made at the same time (one per hand; paired by the recording number in the file
name). So each decoder's errors are averaged per block, and `protocol.session_bootstrap_ci` gives a 95% interval,
both for each decoder and for its **paired difference** from the per-recording median pose. **What to look for:** a
difference whose interval lies entirely below 0 is a real improvement for this person.

In [ ]:
block_of_used = protocol.session_of_rec(stage_of_rec, filename=filename)[used]
per_block = {k: np.array([np.mean(np.array(v)[block_of_used == b]) for b in np.unique(block_of_used)])
             for k, v in within.items()}
ci = {k: protocol.session_bootstrap_ci(v) for k, v in per_block.items()}
ci_diff = {k: protocol.session_bootstrap_ci(v - per_block["median pose"]) for k, v in per_block.items()}
for k in per_block:
    print(f"{k:24s} {ci[k][0]:6.2f}° [{ci[k][1]:5.2f}, {ci[k][2]:5.2f}]   minus median: "
          f"{ci_diff[k][0]:+6.2f}° [{ci_diff[k][1]:+5.2f}, {ci_diff[k][2]:+5.2f}]")
fig, ax = plt.subplots(figsize=(8, 3.5))
names = list(per_block)
ax.errorbar(range(len(names)), [ci_diff[k][0] for k in names],
            yerr=[[ci_diff[k][0] - ci_diff[k][1] for k in names], [ci_diff[k][2] - ci_diff[k][0] for k in names]],
            fmt="o", capsize=4)
ax.axhline(0, color="k", lw=0.8); ax.set_xticks(range(len(names)), names, rotation=15)
ax.set_ylabel("error minus median pose (°)"); ax.set_title("Within recordings: 95% CI over 15 blocks")
plt.tight_layout(); plt.show()

✍️ **Q15.** Rank the decoders within recordings. Did the fewer-parameter physical model beat the free Kalman + velocity model, as you predicted in Q14? How far is the physical decoder above its shifted-EMG null, and what does that gap measure?

> *Your answer:*

---
## Task 14 · Generalise to a stage the decoder has never seen
Within a recording, a decoder can lean on the recording's own average posture (the per-recording median is already
15°, versus 17° for one median for everything). The harder and more relevant test, and the closest one-person proxy
for "a new user", is a **new movement**: fit on every other stage and decode a held-out one.

`protocol.leave_one_stage_out(stage_of_rec, rec, None, filename=filename)` gives 14 folds. A fold's test set is a
stage **plus anything recorded at the same time**, so simultaneous data never leaks into training. (The two
free-style blocks share their recordings with an idle "unconstrained" hand, so they form one fold.)

Within each fold, fit **one** model of each kind on all training frames, then decode every held-out recording from its
own start. The median-pose baseline here is the training median (the held-out stage has no frames of its own).

*Caveat:* the synergies, eigenpostures, τ and delay were chosen on the training split of **all** recordings, held-out
stage included. Notebook 03 showed the synergies transfer across stages almost perfectly, so the leak is small; the
⚠️ Methodology cell lists it.

In [ ]:
loso: dict[str, list[float]] = {k: [] for k in ["median pose", "static log-EMG", "Kalman + velocity", "physical"]}
fold_names: list[str] = []
for name, tr, te in protocol.leave_one_stage_out(stage_of_rec, rec, None, filename=filename):
    fit_f, score_f = tr & pose_valid, te & pose_valid
    if score_f.sum() < 50:
        continue
    fold_names.append(name)
    # TODO: median pose: np.median(pose[fit_f], axis=0) vs pose[score_f]
    # TODO: static log-EMG fitted on fit_f, scored on score_f
    # TODO: fit ONE Kalman + velocity model and ONE physical model on fit_f (outside the loop over recordings!)
    # TODO: for each recording r in te: dec = te & (rec == r); decode LE[dec] with both; store the predictions in
    #       full-length arrays pred_kv, pred_c (np.full_like(pose, np.nan)); then score each on score_f
    ...

for k, e in loso.items():
    print(f"{k:18s} {np.mean(e):5.2f}°  (over {len(e)} held-out stages)")

#### 💡 Stuck on Task 14? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. Fitting is the slow part (the physical and Kalman fits use about 80,000 frames), so fit once per fold.
2. The Kalman filters must restart at every recording, because each recording starts from an unknown hand. Decode one
   held-out recording at a time and write the result into a full-length array.
3. Score with `pred[score_f]` against `pose[score_f]`.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.unique(rec[te])` → the recordings in the held-out fold.
- Writing a block back: `pred[dec] = block` (rows in the same order as `np.flatnonzero(dec)`).

```python
pred = np.full((5, 1), np.nan); dec = np.array([0, 1, 1, 0, 0], bool)
pred[dec] = [[7.0], [8.0]]; pred.ravel()          # [nan, 7., 8., nan, nan]
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
loso["median pose"].append(ref.angular_error(pose[score_f], np.median(pose[fit_f], axis=0)))
    th = ref.least_squares(ref.add_bias(LE[fit_f]), pose[fit_f])
    loso["static log-EMG"].append(ref.angular_error(pose[score_f], ref.add_bias(LE[____]) @ th))
    m_kv = ref.fit_kalman_velocity(pose, LE, fit_f, rec)
    m_c = fit_physical(Qs, A_act, LE, ____, rec)
    pred_kv, pred_c = np.full_like(pose, np.nan), np.full_like(pose, np.nan)
    for r in np.unique(rec[te]):
        dec = te & (rec == r)
        pred_kv[dec] = ref.kalman_velocity_decode(m_kv, LE[dec])
        pred_c[dec] = to_angles(physical_decode(m_c, LE[____]))
    loso["Kalman + velocity"].append(ref.angular_error(pose[score_f], pred_kv[score_f]))
    loso["physical"].append(ref.angular_error(pose[score_f], pred_c[____]))
```

</details>

In [ ]:
# ✅ Check Task 14 (the first fold against the reference)
name0, tr0, te0 = next(iter(protocol.leave_one_stage_out(stage_of_rec, rec, None, filename=filename)))
assert fold_names[0] == name0
fit0, sc0 = tr0 & pose_valid, te0 & pose_valid
m0 = ref.fit_physical(Qs, A_act, LE, fit0, rec)
p0 = np.full_like(pose, np.nan)
for r in np.unique(rec[te0]):
    dec = te0 & (rec == r); p0[dec] = to_angles(ref.physical_decode(m0, LE[dec]))
assert abs(loso["physical"][0] - ref.angular_error(pose[sc0], p0[sc0])) < 1e-6, "physical, first fold"
assert all(len(v) == len(fold_names) for v in loso.values())
print(f"✅ {len(fold_names)} held-out stages decoded")

### 📊 Unseen stages: paired differences from the median pose
Each fold is already an independent unit (a stage plus its simultaneous recordings), so the CI resamples folds.
**What to look for:** compare this with the within-recording figure. Which decoders keep their advantage on a
movement they have never seen?

In [ ]:
loso_diff = {k: protocol.session_bootstrap_ci(np.array(v) - np.array(loso["median pose"])) for k, v in loso.items()}
for k in loso:
    m_, lo_, hi_ = protocol.session_bootstrap_ci(np.array(loso[k]))
    print(f"{k:18s} {m_:6.2f}° [{lo_:5.2f}, {hi_:5.2f}]   minus median: "
          f"{loso_diff[k][0]:+6.2f}° [{loso_diff[k][1]:+5.2f}, {loso_diff[k][2]:+5.2f}]")
fig, ax = plt.subplots(figsize=(7, 3.5))
names = list(loso)
ax.errorbar(range(len(names)), [loso_diff[k][0] for k in names],
            yerr=[[loso_diff[k][0] - loso_diff[k][1] for k in names], [loso_diff[k][2] - loso_diff[k][0] for k in names]],
            fmt="o", capsize=4, color="C1")
ax.axhline(0, color="k", lw=0.8); ax.set_xticks(range(len(names)), names, rotation=15)
ax.set_ylabel("error minus median pose (°)"); ax.set_title(f"Unseen stages: 95% CI over {len(fold_names)} folds")
plt.tight_layout(); plt.show()

✍️ **Q16.** On unseen stages, do the stateful decoders (Kalman + velocity, physical) beat the median pose? If they do *worse*, suggest why a model with dynamics can be hurt by a new movement more than a static map is (hint: what does the filter believe about where the hand rests, and where does that belief come from?).

> *Your answer:*

---
# Part D · Shared dynamics around a per-task resting posture
**The failure this targets.** On unseen stages (Task 14), the stateful decoders did *worse* than the median pose. A
Kalman filter centred on the training data's mean posture keeps pulling its estimate back toward that mean, but a new
task (a fist, a pinch, a pointing hand) rests somewhere else. The dynamics of *how the hand moves* might be shared
across tasks, while *where it rests* is task-specific: an equilibrium point, in Feldman's sense.

**The model.** Centre every recording on **its own** mean pose and mean log-EMG, then fit **one** Kalman + velocity
model on all training recordings together. The dynamics (`A`, `Q`) and the EMG map (`H`, `R`) are shared; the resting
posture is an **offset** added back at decoding time:

$$\hat z(t) = \text{offset}_{\text{pose}} + \text{Kalman}\big(\,y(t) - \text{offset}_{\text{EMG}}\big)$$

Where does a new task's offset come from? Two options, both honest for an unseen stage:
1. **Calibration:** record the first 5 s of the new task *with* the pose (e.g. a camera), use their means. Then score
   only the frames after those 5 s.
2. **From the EMG alone:** guess the resting posture from the task's mean EMG, by a ridge regression fitted across the
   training recordings (one row per recording: mean log-EMG → mean pose). No pose labels needed.

**The honest baseline changes too.** With 5 s of labelled pose, the simplest "decoder" is the **median pose of those
5 s** (the *calibration median*). A model only shows it reads the EMG if it beats *that*.

## Task 15 · Per-recording centring and a shared Kalman + velocity model

In [ ]:
def recording_means(X: np.ndarray, rec: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Mean of X over each recording's mask frames.

    Parameters
    ----------
    X : (N, D) float;  rec : (N,) int recording ids 0 … R−1;  mask : (N,) bool

    Returns
    -------
    (R, D) float, R = rec.max() + 1. Row r = mean of X[mask & (rec == r)]; NaN row if that recording has no mask frames.
    """
    # TODO
    raise NotImplementedError


def velocity_state(Zc: np.ndarray, rec: np.ndarray) -> np.ndarray:
    """State [z, v] with v(t) = z(t) − z(t−1), set to 0 at the first frame of every recording.

    Zc : (N, n) centred pose (or latents). Returns (N, 2n).
    """
    # TODO
    raise NotImplementedError


def fit_kv(Z: np.ndarray, Y: np.ndarray, fit_mask: np.ndarray, rec: np.ndarray, per_recording: bool
           ) -> dict[str, np.ndarray]:
    """One Kalman + velocity model (state [z, v], observation Y) fitted on all fit_mask frames.

    Parameters
    ----------
    Z : (N, n) pose (or latents);  Y : (N, m) observations (log EMG);  fit_mask : (N,) bool;  rec : (N,) int
    per_recording : False → centre Z and Y with their pooled fit_mask means (one resting posture for all tasks);
                    True  → centre each recording with ITS OWN fit_mask means (recording_means), NaN → 0

    Returns
    -------
    {"A", "Q", "H", "R", "P0", "mu_z", "mu_y"}:
      state st = velocity_state(centred Z, rec);  fv = fit_mask frames whose previous frame is also fit_mask and in
      the same recording;  (Zr, _, Zn) = ref.make_triples(st, centred Y, rec, fv);
      A, Q, H, R = ref.fit_kalman(Zr, Zn, st[fv], centred Y[fv])  (fit_kalman already stabilises A);
      P0 = np.cov(st[fv].T, bias=True);  mu_z, mu_y = the POOLED fit_mask means (in both modes)
    """
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 15? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. `recording_means`: an array full of NaN of shape (R, D); for each r, fill row r if any frame of r is in the mask.
2. `velocity_state`: row differences with a zero row on top; then set the velocity to 0 wherever `rec` changes
   (the first frame of each recording); stack `[Zc, v]` side by side.
3. `fit_kv`: compute the pooled means first. Centre either with them or with each recording's own means
   (`recording_means(...)[rec]` gives every frame its recording's row). Then follow the docstring line by line: it is
   Part B's extra, with the centring generalised.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.full((R, D), np.nan)` makes the empty table; `np.nan_to_num(a)` turns NaN into 0.
- `M[rec]` with `M` (R, D) and `rec` (N,) → (N, D): each frame gets its recording's row.
- `first = np.r_[True, rec[1:] != rec[:-1]]` marks each recording's first frame.

```python
rec = np.array([0, 0, 1, 1]); Z = np.array([[1.], [3.], [10.], [11.]])
v = np.vstack([np.zeros((1, 1)), np.diff(Z, axis=0)])   # [[0], [2], [7], [1]]
v[np.r_[True, rec[1:] != rec[:-1]]] = 0                # [[0], [2], [0], [1]]: no jump across recordings
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def recording_means(X, rec, mask):
    R = int(rec.max()) + 1
    out = np.full((R, X.shape[1]), np.nan)
    for r in range(R):
        m = mask & (rec == ____)
        if m.any():
            out[r] = X[m].mean(axis=____)
    return out


def velocity_state(Zc, rec):
    v = np.vstack([np.zeros((1, Zc.shape[1])), np.diff(Zc, axis=0)])
    v[np.r_[True, rec[1:] != rec[:-1]]] = ____
    return np.hstack([Zc, v])


def fit_kv(Z, Y, fit_mask, rec, per_recording):
    mu_z, mu_y = Z[fit_mask].mean(axis=0), Y[fit_mask].mean(axis=0)
    if per_recording:
        Zc = Z - np.nan_to_num(recording_means(Z, rec, fit_mask)[____])
        Yc = Y - np.nan_to_num(recording_means(Y, rec, fit_mask)[____])
    else:
        Zc, Yc = Z - mu_z, Y - mu_y
    st = velocity_state(Zc, rec)
    fv = np.r_[False, fit_mask[1:] & fit_mask[:-1] & (rec[1:] == rec[:-1])]
    Zr, _, Zn = ref.make_triples(st, Yc, rec, ____)
    A, Q, H, R = ref.fit_kalman(Zr, Zn, st[fv], Yc[fv])
    return {"A": ____, "Q": Q, "H": H, "R": R,
            "P0": np.cov(st[fv].T, bias=True), "mu_z": mu_z, "mu_y": mu_y}
```

</details>

In [ ]:
# ✅ Check Task 15
fit_all = train & pose_valid
assert np.allclose(recording_means(pose, rec, fit_all), ref.recording_means(pose, rec, fit_all), equal_nan=True), \
    "recording_means differs"
toy_rec = np.array([0, 0, 1, 1]); toy_Z = np.array([[1.0], [3.0], [10.0], [11.0]])
assert np.allclose(velocity_state(toy_Z, toy_rec), ref.velocity_state(toy_Z, toy_rec)), \
    "velocity_state differs: is the velocity 0 at the first frame of each recording?"
m_shared = fit_kv(pose, LE, fit_all, rec, per_recording=True)
r_shared = ref.fit_kv(pose, LE, fit_all, rec, per_recording=True)
for k in ["A", "Q", "H", "R", "P0"]:
    assert np.allclose(m_shared[k], r_shared[k], atol=1e-6), f"{k} differs from the reference"
print(f"✅ Shared model fitted on {fit_all.sum()} frames of {len(np.unique(rec[fit_all]))} recordings. "
      f"Largest |eigenvalue| of A = {np.abs(np.linalg.eigvals(m_shared['A'])).max():.3f}")

✍️ **Q17.** Centring each recording on its own mean removes *where* each task rests from the data the dynamics
are fitted on. What does the shared `A` then describe? Why might a per-task offset fix the unseen-stage failure of
Task 14, and what could it *not* fix?

> *Your answer:*

## Task 16 · Decoding with an offset, and guessing the offset from the EMG
Two small functions:
- `kv_decode`: filter `Y_block − y_offset` from a zero state (the offset already says where the hand rests), then add
  `z_offset` back to the decoded pose.
- `offset_from_emg`: one row per training recording (its mean log-EMG → its mean pose), a ridge regression across those
  rows, applied to the new task's mean log-EMG. Centre both sides first so the intercept isn't penalised (as in
  notebook 05's ridge).

Then the **within-recording** test: one shared model for all recordings, each recording decoded with its own training
means as offsets. Compare with Part B's per-recording Kalman + velocity, which fits a whole model per recording.

In [ ]:
def kv_decode(model: dict[str, np.ndarray], Y_block: np.ndarray, z_offset: np.ndarray, y_offset: np.ndarray
              ) -> np.ndarray:
    """Decode one contiguous block.

    Y_block : (T, m);  z_offset : (n,) resting posture;  y_offset : (m,) resting EMG.
    Returns (T, n): the first n state columns of ref.kalman_filter(Y_block − y_offset, A, Q, H, R, zeros(2n), P0),
    plus z_offset. (n = model["A"].shape[0] // 2.)
    """
    # TODO
    raise NotImplementedError


def offset_from_emg(emg_means: np.ndarray, pose_means: np.ndarray, new_emg_mean: np.ndarray,
                    alpha: float = 10.0) -> np.ndarray:
    """Guess a resting posture from a mean EMG.

    emg_means : (R, m), pose_means : (R, n): one row per TRAINING recording;  new_emg_mean : (m,).
    Centre both with their row means, ridge B = solve(Xcᵀ Xc + alpha I, Xcᵀ Yc), return (new − x̄) B + ȳ, shape (n,).
    """
    # TODO
    raise NotImplementedError


# Within recordings: one shared model, each recording decoded with its own training means as offsets.
pose_means_tr, emg_means_tr = recording_means(pose, rec, fit_all), recording_means(LE, rec, fit_all)
within["shared KV + task offset"] = []
for r in used:                                         # the same recordings, in the same order, as Task 13
    dec_r = test & (rec == r)
    # TODO: pred = kv_decode(m_shared, LE[dec_r], pose_means_tr[r], emg_means_tr[r]);
    #       append ref.angular_error(pose[dec_r & pose_valid], pred[pose_valid[dec_r]]) to within["shared KV + task offset"]
    ...
print(f"shared KV + task offset: {np.mean(within['shared KV + task offset']):.2f}°   "
      f"(per-recording Kalman + velocity: {np.mean(within['Kalman + velocity']):.2f}°, "
      f"median pose: {np.mean(within['median pose']):.2f}°)")

#### 💡 Stuck on Task 16? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. `kv_decode`: work out n, run the filter on the offset-removed observations from a zero state, keep the first n
   columns, add the pose offset.
2. `offset_from_emg`: it is ridge regression with R ≈ 26 rows (training recordings) and m = 16 columns; follow the
   docstring.
3. The loop: two lines, exactly as in the comment.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `ref.kalman_filter(Y, A, Q, H, R, z0, P0)` returns `(Z_hat, P_trace)`; you only need `Z_hat`.
- Ridge with centring: `Xc = X - X.mean(0)`, `Yc = Y - Y.mean(0)`,
  `B = np.linalg.solve(Xc.T @ Xc + alpha * np.eye(Xc.shape[1]), Xc.T @ Yc)`.

```python
X = np.array([[0.], [1.], [2.]]); Y = np.array([[1.], [3.], [5.]])
Xc, Yc = X - X.mean(0), Y - Y.mean(0)
B = np.linalg.solve(Xc.T @ Xc + 0.0 * np.eye(1), Xc.T @ Yc)   # [[2.]]
(np.array([3.]) - X.mean(0)) @ B + Y.mean(0)                   # [7.]
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def kv_decode(model, Y_block, z_offset, y_offset):
    n = model["A"].shape[0] // 2
    Zh, _ = ref.kalman_filter(Y_block - ____, model["A"], model["Q"], model["H"], model["R"],
                              np.zeros(2 * n), model["P0"])
    return Zh[:, :____] + z_offset


def offset_from_emg(emg_means, pose_means, new_emg_mean, alpha=10.0):
    ex, pz = emg_means.mean(axis=0), pose_means.mean(axis=0)
    Xc, Yc = emg_means - ex, pose_means - pz
    B = np.linalg.solve(Xc.T @ Xc + alpha * np.eye(Xc.shape[1]), Xc.T @ ____)
    return (new_emg_mean - ____) @ B + pz

for r in used:
    dec_r = test & (rec == r)
    pred = kv_decode(m_shared, LE[dec_r], pose_means_tr[r], emg_means_tr[r])
    within["shared KV + task offset"].append(ref.angular_error(pose[dec_r & pose_valid], pred[pose_valid[____]]))
```

</details>

In [ ]:
# ✅ Check Task 16
r0 = used[0]; dec0 = test & (rec == r0)
assert np.allclose(kv_decode(m_shared, LE[dec0], pose_means_tr[r0], emg_means_tr[r0]),
                   ref.kv_decode(r_shared, LE[dec0], pose_means_tr[r0], emg_means_tr[r0]), atol=1e-6), "kv_decode differs"
ok_r = ~np.isnan(pose_means_tr[:, 0])
assert np.allclose(offset_from_emg(emg_means_tr[ok_r], pose_means_tr[ok_r], emg_means_tr[r0]),
                   ref.offset_from_emg(emg_means_tr[ok_r], pose_means_tr[ok_r], emg_means_tr[r0]), atol=1e-6), \
    "offset_from_emg differs"
assert len(within["shared KV + task offset"]) == len(used), "one error per scored recording"
print("✅ Task 16 works")

## Task 17 · Unseen stages, with 5 s of calibration
The real test of Part D. For every leave-one-stage-out fold (`protocol.leave_one_stage_out`, 14 folds):
- the first **5 s** (250 frames) of each held-out recording are the **calibration window**
  (`ref.calibration_masks(te, rec, CAL)` returns it and the frames after it);
- every decoder is scored **only on the frames after** the calibration window, so all of them are compared on the
  same frames;
- each held-out recording is decoded from its start (the filter may use the calibration window's EMG to settle).

| Decoder | Uses the pose of the 5 s? | What it is |
|---|---|---|
| median pose | no | the training stages' median |
| **calibration median** | yes | the median pose of the 5 s: the honest baseline |
| static log-EMG | no | Part C's static least-squares map, fitted on the training stages |
| KV, pooled | no | Kalman + velocity centred on the training stages' mean (Task 14's failure) |
| **KV + task offset (5 s)** | yes | Task 15's shared model; offsets = mean pose and mean log-EMG of the 5 s |
| **KV + task offset from EMG** | no | the same, pose offset = `offset_from_emg` of the 5 s' mean log-EMG |
| KV + task offset, shifted EMG | yes | the null: the same as "(5 s)" with EMG shifted in time within each recording |

In [ ]:
CAL = 250                                                   # 5 s at 50 frames/s
rng = np.random.default_rng(SEED)
LE_shift = protocol.circular_shift_null(LE, rec, rng)       # null: EMG misaligned with the pose
CAL_KEYS = ["median pose", "calibration median", "static log-EMG", "KV, pooled", "KV + task offset (5 s)",
            "KV + task offset from EMG", "KV + task offset, shifted EMG"]
loso_cal: dict[str, list[float]] = {k: [] for k in CAL_KEYS}
cal_folds: list[str] = []

for name, tr, te in protocol.leave_one_stage_out(stage_of_rec, rec, None, filename=filename):
    fit_f = tr & pose_valid
    cal, after = ref.calibration_masks(te, rec, CAL)
    score_f = after & pose_valid
    if score_f.sum() < 50:
        continue
    cal_folds.append(name)
    pmed = np.median(pose[fit_f], axis=0)
    th = ref.least_squares(ref.add_bias(LE[fit_f]), pose[fit_f])
    m_pool = fit_kv(pose, LE, fit_f, rec, per_recording=False)
    m_task = fit_kv(pose, LE, fit_f, rec, per_recording=True)
    m_null = fit_kv(pose, LE_shift, fit_f, rec, per_recording=True)
    pm_f, em_f = recording_means(pose, rec, fit_f), recording_means(LE, rec, fit_f)
    ok_f = ~np.isnan(pm_f[:, 0])
    pred = {k: np.full_like(pose, np.nan) for k in CAL_KEYS}
    pred["median pose"][te] = pmed
    pred["static log-EMG"][te] = ref.add_bias(LE[te]) @ th
    for r in np.unique(rec[te]):
        dec, c = te & (rec == r), cal & (rec == r)
        cv = c & pose_valid
        p_off = pose[cv].mean(axis=0) if cv.sum() > 10 else pmed     # resting posture from the 5 s
        e_off = LE[c].mean(axis=0)                                  # resting EMG from the 5 s
        # TODO: pred["calibration median"][dec] = the median pose of the 5 s (or pmed if cv.sum() <= 10)
        # TODO: pred["KV, pooled"][dec]        = kv_decode(m_pool, LE[dec], m_pool["mu_z"], m_pool["mu_y"])
        # TODO: pred["KV + task offset (5 s)"][dec]    = kv_decode(m_task, LE[dec], p_off, e_off)
        # TODO: pred["KV + task offset from EMG"][dec] = kv_decode(m_task, LE[dec],
        #                                                   offset_from_emg(em_f[ok_f], pm_f[ok_f], e_off), e_off)
        # TODO: pred["KV + task offset, shifted EMG"][dec] = kv_decode(m_null, LE_shift[dec], p_off, LE_shift[c].mean(axis=0))
        ...
    for k in CAL_KEYS:
        loso_cal[k].append(ref.angular_error(pose[score_f], pred[k][score_f]))

for k, e in loso_cal.items():
    print(f"{k:30s} {np.mean(e):5.2f}°  (over {len(e)} held-out stages)")

#### 💡 Stuck on Task 17? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

Everything outside the inner loop is given. Inside it, each TODO is one line: build the prediction for this held-out
recording's frames `dec`, using the models fitted once per fold above.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.median(pose[cv], axis=0)` → (20,), and NumPy broadcasts it to every row of `pred[...][dec]`.
- `kv_decode(model, Y_block, z_offset, y_offset)` from Task 16.

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
pred["calibration median"][dec] = np.median(pose[cv], axis=0) if cv.sum() > 10 else ____
        pred["KV, pooled"][dec] = kv_decode(m_pool, LE[dec], m_pool["mu_z"], m_pool["____"])
        pred["KV + task offset (5 s)"][dec] = kv_decode(m_task, LE[dec], ____, ____)
        pred["KV + task offset from EMG"][dec] = kv_decode(m_task, LE[dec],
                                                           offset_from_emg(em_f[ok_f], pm_f[ok_f], e_off), e_off)
        pred["KV + task offset, shifted EMG"][dec] = kv_decode(m_null, LE_shift[dec], p_off, LE_shift[c].mean(axis=0))
```

</details>

In [ ]:
# ✅ Check Task 17 (the first fold against the reference)
name0, tr0, te0 = next(iter(protocol.leave_one_stage_out(stage_of_rec, rec, None, filename=filename)))
assert cal_folds[0] == name0
cal0, after0 = ref.calibration_masks(te0, rec, CAL); sc0 = after0 & pose_valid
m0 = ref.fit_kv(pose, LE, tr0 & pose_valid, rec, per_recording=True)
p0 = np.full_like(pose, np.nan)
for r in np.unique(rec[te0]):
    dec, c = te0 & (rec == r), cal0 & (rec == r); cv = c & pose_valid
    po = pose[cv].mean(axis=0) if cv.sum() > 10 else np.median(pose[tr0 & pose_valid], axis=0)
    p0[dec] = ref.kv_decode(m0, LE[dec], po, LE[c].mean(axis=0))
assert abs(loso_cal["KV + task offset (5 s)"][0] - ref.angular_error(pose[sc0], p0[sc0])) < 1e-6, "first fold differs"
assert all(len(v) == len(cal_folds) for v in loso_cal.values())
print(f"✅ {len(cal_folds)} held-out stages decoded")

### 📊 Unseen stages: every decoder against the calibration median
Each dot is the mean over folds of *decoder error minus calibration-median error*, with a 95% CI over the folds.
**What to look for:** below 0 means the decoder beats "just use the median pose of the first 5 s". Is the offset model
below the pooled one? Below its shifted-EMG null? Below 0 at all?

In [ ]:
ci_cal = {k: protocol.session_bootstrap_ci(np.array(v) - np.array(loso_cal["calibration median"]))
          for k, v in loso_cal.items()}
for k, (m_, lo_, hi_) in ci_cal.items():
    print(f"{k:30s} minus calibration median: {m_:+6.2f}° [{lo_:+5.2f}, {hi_:+5.2f}]")
fig, ax = plt.subplots(figsize=(9, 3.5))
names = list(ci_cal)
ax.errorbar(range(len(names)), [ci_cal[k][0] for k in names],
            yerr=[[ci_cal[k][0] - ci_cal[k][1] for k in names], [ci_cal[k][2] - ci_cal[k][0] for k in names]],
            fmt="o", capsize=4, color="C2")
ax.axhline(0, color="k", lw=0.8); ax.set_xticks(range(len(names)), names, rotation=20, ha="right", fontsize=8)
ax.set_ylabel("minus calibration median (°)"); ax.set_title(f"Unseen stages, scored after 5 s: CI over {len(cal_folds)} folds")
plt.tight_layout(); plt.show()

✍️ **Q18.** Compare "KV, pooled" with "KV + task offset (5 s)": how much did the offset fix? Then compare the offset
model with the **calibration median** and with its shifted-EMG null. Does the EMG add anything once the resting
posture is known? And does `offset_from_emg` recover the resting posture from the EMG alone, or is it worse than
no offset? What does that say about how much of "which posture" the wrist EMG carries?

> *Your answer:*

---
# Part E · Dynamics in the shared EMG–pose subspace
Notebook 06 found that the EMG directions that predict the pose form a **2–3 dimensional subspace** (reduced-rank
regression), which is *not* the subspace of the biggest EMG synergies. So instead of 20 joint angles (Parts B, D),
model the dynamics of just the **r = 3 shared latents**:

- **pose side:** `z(t) = (pose(t) − ȳ) V`, shape (N, 3): the pose directions the EMG can predict (V from `rrr_fit`);
- **EMG side:** `s(t)`, shape (N, 3): the EMG projected on the directions the reduced-rank map uses
  (`ref06.shared_latents`), from log EMG 100 and 200 ms earlier (`X_lag`).

Then a Kalman + velocity model whose state is `[z, ż]` (6 numbers) and whose observation is `s` (3 numbers), with the
same per-task offsets as Part D. The decoded latents go back to 20 angles with `offset + z V'`. Everything the 3
latents can't represent is filled in by the resting posture.

If the dynamics really live in these few dimensions, this tiny model (about 6 × 6 dynamics numbers instead of
40 × 40) should match Part D's 20-D model, and might generalise better.

## Task 18 · The two sides of the shared subspace

In [ ]:
RANK = 3


def latent_spaces(X: np.ndarray, pose: np.ndarray, fit_mask: np.ndarray, rank: int, alpha: float = 1.0
                  ) -> tuple[np.ndarray, np.ndarray, dict[str, np.ndarray]]:
    """Both sides of the shared EMG–pose subspace, fitted on fit_mask frames.

    Returns
    -------
    S   : (N, rank) EMG-side latents, ref06.shared_latents(X, B_r, x_mean, rank)
    Zq  : (N, rank) pose-side latents, (pose − y_mean) @ V
    sub : {"B_r", "x_mean", "y_mean", "V"} from ref06.rrr_fit(X[fit_mask], pose[fit_mask], rank, alpha)
    """
    # TODO
    raise NotImplementedError


def latent_to_pose(z_lat: np.ndarray, sub: dict[str, np.ndarray], pose_offset: np.ndarray) -> np.ndarray:
    """(T, rank) latents centred on a task's resting posture → (T, 20) angles: pose_offset + z_lat @ Vᵀ."""
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 18? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. Fit notebook 06's reduced-rank map on the fit frames only.
2. EMG side: project the (centred) EMG features on the directions the map uses (one ready-made call).
3. Pose side: centre the pose with the map's `y_mean` and multiply by `V`.
4. `latent_to_pose` is one line: undo the projection (`@ V.T`) and add the offset.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `B_r, x_mean, y_mean, V = ref06.rrr_fit(X, Y, rank, alpha)`; `V` is (20, rank) with orthonormal columns.
- `ref06.shared_latents(X, B_r, x_mean, rank)` → (N, rank).
- Because V's columns are orthonormal, `z @ V.T` maps latents back into the 20-D pose space.

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def latent_spaces(X, pose, fit_mask, rank, alpha=1.0):
    B_r, x_mean, y_mean, V = ref06.rrr_fit(X[____], pose[____], rank, alpha)
    S = ref06.shared_latents(X, B_r, x_mean, rank)
    Zq = (pose - ____) @ V
    return S, Zq, {"B_r": B_r, "x_mean": x_mean, "y_mean": y_mean, "V": V}


def latent_to_pose(z_lat, sub, pose_offset):
    return pose_offset + z_lat @ sub["____"].T
```

</details>

In [ ]:
# ✅ Check Task 18
S_all, Zq_all, sub_all = latent_spaces(X_lag, pose, fit_all, RANK)
rS, rZ, rsub = ref.latent_spaces(X_lag, pose, fit_all, RANK)
assert S_all.shape == (len(pose), RANK) and Zq_all.shape == (len(pose), RANK)
assert np.allclose(np.abs(S_all), np.abs(rS), atol=1e-6) and np.allclose(np.abs(Zq_all), np.abs(rZ), atol=1e-6), \
    "latents differ from the reference (up to sign)"
z_back = latent_to_pose(Zq_all[:5], sub_all, sub_all["y_mean"])
assert np.allclose(z_back, ref.latent_to_pose(Zq_all[:5], sub_all, sub_all["y_mean"])), "latent_to_pose differs"
lost = ref.angular_error(pose[fit_all], latent_to_pose(Zq_all[fit_all], sub_all, sub_all["y_mean"]))
print(f"✅ Shared subspace works. Keeping only the {RANK} pose-side latents (around the overall mean) loses "
      f"{lost:.1f}° on the training frames")

### 📊 The three shared latents over time
Pose-side latents (solid) and EMG-side latents (dashed) for one recording's test block. **What to look for:** do the
EMG-side latents rise and fall *before* the pose-side ones? Which latent tracks the pose best?

In [ ]:
r_pl = used[1]; m_pl = test & (rec == r_pl); t_pl = np.arange(m_pl.sum()) / 50
fig, ax = plt.subplots(RANK, 1, figsize=(14, 6), sharex=True)
for i in range(RANK):
    zi = np.where(pose_valid[m_pl], Zq_all[m_pl, i], np.nan)
    si = S_all[m_pl, i]
    si = (si - si.mean()) / (si.std() + 1e-9) * np.nanstd(zi) + np.nanmean(zi)   # rescaled to overlay
    ax[i].plot(t_pl, zi, color="k", lw=1.2, label="pose-side latent")
    ax[i].plot(t_pl, si, color=f"C{i}", lw=1, ls="--", label="EMG-side latent (rescaled)")
    ax[i].set_ylabel(f"latent {i + 1}")
ax[0].legend(loc="upper right", fontsize=8); ax[-1].set_xlabel("time into the test block (s)")
ax[0].set_title(f"Recording {r_pl} ({stage_of_rec[r_pl]})")
plt.tight_layout(); plt.show()

## Task 19 · Decode through the shared latents
Two settings, as before:
- **within recordings:** one shared latent model (`fit_kv` on `Zq_all`, `S_all`, per-recording centring), each
  recording decoded with its own training means as offsets; the pose offset is the recording's 20-D mean pose;
- **unseen stages, 5 s calibration:** inside each fold refit the subspace *and* the latent model on the training
  stages only; offsets from the 5 s, as in Task 17. Also score the **static** reduced-rank map (no dynamics,
  `ref06.rrr_predict`) on the same frames.

`latent_to_pose(z, sub, pose_offset)` turns decoded latents back into angles. Note `kv_decode(..., z_offset=0, ...)`
returns latents *around the resting posture*, which is exactly what `latent_to_pose` expects.

In [ ]:
# Within recordings (the same recordings and order as Task 13)
m_lat = fit_kv(Zq_all, S_all, fit_all, rec, per_recording=True)
s_means_tr = recording_means(S_all, rec, fit_all)
within["RRR static (rank 3)"] = []
within["shared-subspace KV + task offset"] = []
for r in used:
    dec_r = test & (rec == r); sc_r = dec_r & pose_valid
    within["RRR static (rank 3)"].append(ref.angular_error(
        pose[sc_r], ref06.rrr_predict(X_lag[sc_r], sub_all["B_r"], sub_all["x_mean"], sub_all["y_mean"])))
    # TODO: z = kv_decode(m_lat, S_all[dec_r], np.zeros(RANK), s_means_tr[r])
    # TODO: pred = latent_to_pose(z, sub_all, pose_means_tr[r]); score pred[pose_valid[dec_r]] against pose[sc_r]
    ...

# Unseen stages, 5 s calibration (the same folds and scored frames as Task 17)
loso_cal["RRR static (rank 3)"] = []
loso_cal["shared-subspace KV + task offset (5 s)"] = []
for name, tr, te in protocol.leave_one_stage_out(stage_of_rec, rec, None, filename=filename):
    fit_f = tr & pose_valid
    cal, after = ref.calibration_masks(te, rec, CAL)
    score_f = after & pose_valid
    if score_f.sum() < 50:
        continue
    S_f, Zq_f, sub_f = latent_spaces(X_lag, pose, fit_f, RANK)
    m_f = fit_kv(Zq_f, S_f, fit_f, rec, per_recording=True)
    pred_s, pred_k = np.full_like(pose, np.nan), np.full_like(pose, np.nan)
    pred_s[te] = ref06.rrr_predict(X_lag[te], sub_f["B_r"], sub_f["x_mean"], sub_f["y_mean"])
    for r in np.unique(rec[te]):
        dec, c = te & (rec == r), cal & (rec == r)
        cv = c & pose_valid
        p_off = pose[cv].mean(axis=0) if cv.sum() > 10 else np.median(pose[fit_f], axis=0)
        # TODO: z = kv_decode(m_f, S_f[dec], np.zeros(RANK), S_f[c].mean(axis=0))
        # TODO: pred_k[dec] = latent_to_pose(z, sub_f, p_off)
        ...
    loso_cal["RRR static (rank 3)"].append(ref.angular_error(pose[score_f], pred_s[score_f]))
    loso_cal["shared-subspace KV + task offset (5 s)"].append(ref.angular_error(pose[score_f], pred_k[score_f]))

for k in ["RRR static (rank 3)", "shared-subspace KV + task offset"]:
    print(f"within  {k:42s} {np.mean(within[k]):5.2f}°")
for k in ["RRR static (rank 3)", "shared-subspace KV + task offset (5 s)", "KV + task offset (5 s)", "calibration median"]:
    print(f"unseen  {k:42s} {np.mean(loso_cal[k]):5.2f}°")

#### 💡 Stuck on Task 19? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

Four lines in total, two in each loop, exactly as in the comments: decode the latents around the resting posture,
then map them back to angles with the recording's (or the calibration window's) pose offset, then score.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `kv_decode(model, Y_block, z_offset, y_offset)` with `z_offset = np.zeros(RANK)` returns latents around rest.
- `latent_to_pose(z, sub, pose_offset)` → (T, 20).
- Score like Task 16: `ref.angular_error(pose[sc_r], pred[pose_valid[dec_r]])`.

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
z = kv_decode(m_lat, S_all[dec_r], np.zeros(RANK), s_means_tr[____])
    pred = latent_to_pose(z, sub_all, pose_means_tr[____])
    within["shared-subspace KV + task offset"].append(ref.angular_error(pose[sc_r], pred[pose_valid[dec_r]]))

        z = kv_decode(m_f, S_f[dec], np.zeros(RANK), S_f[c].mean(axis=____))
        pred_k[dec] = latent_to_pose(z, sub_f, ____)
```

</details>

In [ ]:
# ✅ Check Task 19
assert len(within["shared-subspace KV + task offset"]) == len(used)
assert all(len(v) == len(cal_folds) for v in loso_cal.values()), "every unseen-stage decoder needs one error per fold"
r0 = used[0]; dec0 = test & (rec == r0)
rm = ref.fit_kv(rZ, rS, fit_all, rec, per_recording=True)
z0 = ref.kv_decode(rm, rS[dec0], np.zeros(RANK), ref.recording_means(rS, rec, fit_all)[r0])
e0 = ref.angular_error(pose[dec0 & pose_valid], ref.latent_to_pose(z0, rsub, pose_means_tr[r0])[pose_valid[dec0]])
assert abs(within["shared-subspace KV + task offset"][0] - e0) < 1e-6, "first recording differs from the reference"
print("✅ Shared-subspace decoders scored")

✍️ **Q19.** The latent model has a 6-number state; Part D's has 40. Within recordings and on unseen stages, how does
it compare with Part D's 20-D offset model and with the static reduced-rank map? What does that say about how many
dimensions the EMG-driven *dynamics* need, compared with the 2–3 the static map needs (notebook 06)?

> *Your answer:*

---
# Part F · All decoders, with confidence intervals
Ready-made: nothing to fill in. Within recordings, errors are averaged per **block** (the two simultaneous recordings)
and compared with the per-recording median pose; on unseen stages, per **fold** and compared with the calibration
median. 95% bootstrap CIs on the paired differences.

In [ ]:
block_used = protocol.session_of_rec(stage_of_rec, filename=filename)[used]
per_block = {k: np.array([np.mean(np.array(v)[block_used == b]) for b in np.unique(block_used)]) for k, v in within.items()}
print(f"WITHIN RECORDINGS ({len(np.unique(block_used))} blocks)            mean    minus median pose [95% CI]")
for k, v in per_block.items():
    dm, dlo, dhi = protocol.session_bootstrap_ci(v - per_block["median pose"])
    print(f"  {k:42s} {v.mean():6.2f}°   {dm:+6.2f}° [{dlo:+5.2f}, {dhi:+5.2f}]")
print(f"\nUNSEEN STAGES, scored after 5 s ({len(cal_folds)} folds)   mean    minus calibration median [95% CI]")
for k, v in loso_cal.items():
    dm, dlo, dhi = protocol.session_bootstrap_ci(np.array(v) - np.array(loso_cal["calibration median"]))
    print(f"  {k:42s} {np.mean(v):6.2f}°   {dm:+6.2f}° [{dlo:+5.2f}, {dhi:+5.2f}]")

✍️ **Q20.** Rank the structures you added: state (Part B), physics (Part C), a per-task resting posture (Part D),
shared latents (Part E). Which helped within recordings, which on unseen stages, and which didn't help at all? Which
single result most changes what stage 6 (bottlenecks) should test?

> *Your answer:*

---
## ⚠️ Methodology: what this notebook can and can't tell you
Read this before writing the Result. Each row is a reason the numbers could look better (or different) than the truth, and what would test it.

| Issue | Why it matters | What would test it |
|---|---|---|
| **Fifteen blocks, one person** | Recordings come in 15 simultaneous left/right blocks (paired by recording number; two blocks pair OneHandedFreeStyle with the idle hand's 'unconstrained'). Within-recording CIs resample blocks; unseen-stage CIs resample the 14 folds | Stage 7: other people |
| **Per-recording calibration is the easy test** | Parts B and C fit each decoder on the first 80% of the same recording they decode | The unseen-stage results (Tasks 14, 17, 19) are the ones that matter for calibration claims |
| **The 5 s calibration uses the pose** | Part D's best offsets need 5 s of labelled pose from the new task; that is a camera, not EMG alone | The 'from EMG' offset (no labels); longer or shorter windows |
| **Zero-phase envelope** | The 10 Hz envelope already contains about 100 ms of future EMG, so the 'causal' filters and the fitted delay are slightly optimistic | Rebuild the envelope causally (sosfilt) and rerun |
| **Pose smoothing and stabilisation** | Version-2 pose is low-passed at 6 Hz; that removes jitter but also makes velocity so predictable that least squares returned unstable A matrices, shrunk by ref.stabilise | Compare 4 and 10 Hz cut-offs; refit with a ridge penalty instead of eigenvalue shrinking |
| **Shared choices leak a little** | Synergies, eigenpostures, τ and delay, and the within-recording subspace were fitted on all recordings' training frames, including stages held out later | Refit everything inside each fold (Task 19 already does this for the subspace) |
| **Linear and Gaussian** | Every model here is linear with Gaussian noise; hands have joint limits, and EMG noise is signal-dependent | Notebook 07b: switching motifs and a nonlinear RNN |

---
## Result · gate for stage 5
**Question:** Is the hand's movement input-driven, and which structure gives a decoder that beats the honest baselines?

| Measure | Value (mean, 95% CI over sessions) | Baseline / null | Pass? |
|---|---|---|---|
| Rollout error at 25 steps: no change / A only / A and B (Part A) | | | |
| Rollout error at 25 / 50 steps: no change / input off / driven (Part C) | | | |
| Mean damping D on the stored pose (Part C) | | | |
| Within recordings, minus median pose: Kalman + velocity / physical / shared KV + task offset / shared-subspace KV | | | |
| Unseen stages (after 5 s), minus calibration median: KV pooled / KV + task offset (5 s) / from EMG / shifted EMG | | | |
| Unseen stages (after 5 s), minus calibration median: RRR static / shared-subspace KV + task offset | | | |

**Gate:** the EMG-driven model beats the input-off model in free rollouts, **and** a stateful decoder beats the calibration median on unseen stages with a 95% CI that excludes 0.

- **Pass →** carry the best decoder into stage 6 (notebook 08): the bottleneck comparison and calibration curves.
- **Fail →** report which half failed. A decoder that can't beat the median pose of the first 5 s on a new task can't support calibration claims; try the compositional models (07b) and say so before running stage 6.
- **Was your prediction right? Why or why not?**

✍️ **Q21.** Fill in the gate. Which half failed, if any? Look at the gap between within-recording and unseen-stage
results: what is the *smallest* next change you would test?

> *Your answer:*